# The Anatomy of AI · Lecture 27 · Tracing a Thought — the notebook

Ask Gemma-2-2B to finish *Fact: the capital of the state containing Dallas is* and it says **Austin**. The prompt never
names Texas. Does the model think *Dallas → Texas → Austin* in two steps, or did it learn the answer as one lookup?

The lecture built the tool that answers this, an **attribution graph**, and in this notebook you build it yourself:

1. **Gemma, written out.** The model as 30 lines of PyTorch, checked against the library.
2. **A readable copy of each MLP**, the **transcoder**: on Gemma (Google DeepMind's Gemma Scope transcoders), and trained
   by you on GPT-2.
3. **The network as a sum of arrows.** Freeze attention and the norms, and every number inside the model splits exactly
   into arrows from readable features, from the words, and from *error nodes* (what the copy misses). You compute those
   arrows by hand, all 75 million of them, and score the graph.
4. **Dallas → Texas → Austin.** Read the graph, then test it by surgery on the real model: turn Texas down and California
   up, and Gemma says Sacramento.

Every key number is checked against the lecture's: a ✓ means you got what the lecture got. It needs a GPU
(*Runtime → Change runtime type → T4 GPU*) and takes about 15 minutes on free Colab, about a third of it downloading
Gemma (5.2 GB) and its 26 transcoders (7.9 GB).

*One difference from the lecture.* The lecture ran Gemma through the circuit-tracer library in bfloat16 (16-bit
numbers). Here everything is written out and runs in full 32-bit precision, so most numbers agree to within about 1%
rather than to the last digit; each check says how close it has to be.

In [ ]:
# Pinned installs. Free Colab already has torch and torchvision; a machine without them gets the pinned pair.
import importlib.util, os, subprocess, sys
os.environ.update(HF_HUB_DISABLE_PROGRESS_BARS="1", TRANSFORMERS_VERBOSITY="error", TOKENIZERS_PARALLELISM="false",
                  PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True")
def pip(*args):                                   # quiet: pip's output is shown only if an install fails
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check", *args],
                       capture_output=True, text=True)
    if r.returncode:
        raise RuntimeError(r.stdout[-3000:] + r.stderr[-3000:])
if importlib.util.find_spec("torch") is None:
    pip("torch==2.5.1", "torchvision==0.20.1")
pip("numpy==2.0.2", "matplotlib==3.10.0", "pillow==11.1.0", "pyarrow==18.1.0", "requests==2.32.3",
    "transformers==4.46.3", "accelerate==1.1.1", "safetensors==0.4.5", "huggingface_hub==0.26.2")
print("installed")

In [ ]:
import io, json, math, os, random, time, warnings
warnings.filterwarnings("ignore", category=FutureWarning)      # library notices, not results
warnings.filterwarnings("ignore", category=UserWarning)
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F

DEV = "cuda" if torch.cuda.is_available() else "cpu"
print("running on", DEV)
T0 = time.time()

# The course's look: black ink on paper, no colour except real photos.
PAPER, INK, INK2 = "#F1EBDD", "#16130F", "#7A7062"
plt.rcParams.update({
    "figure.facecolor": PAPER, "axes.facecolor": PAPER, "savefig.facecolor": PAPER,
    "axes.edgecolor": INK, "axes.labelcolor": INK, "xtick.color": INK, "ytick.color": INK, "text.color": INK,
    "font.family": "serif", "font.size": 11, "axes.spines.top": False, "axes.spines.right": False,
    "hatch.color": INK, "hatch.linewidth": 0.6, "figure.dpi": 110, "legend.frameon": False,
    "lines.color": INK, "patch.edgecolor": INK, "axes.prop_cycle": plt.cycler(color=[INK, INK2]),
})


def minus(text):
    """Print negative numbers with a real minus sign, as the lecture does."""
    return str(text).replace("-", "\u2212")


def check(name, value, expected, tol=0):
    """Compare a result with the lecture's own number. Prints ✓, or stops the notebook with ✗."""
    if isinstance(expected, (int, float)) and not isinstance(expected, bool):
        ok = abs(float(value) - expected) <= tol
        shown = f"{float(value):.6g}"
    else:
        ok = value == expected
        shown = repr(value)
    extra = f" ± {tol:g}" if isinstance(expected, (int, float)) and not isinstance(expected, bool) and tol else ""
    print(f"{'✓' if ok else '✗'} {name}: {minus(shown)}   (lecture: {minus(expected)}{extra})")
    if not ok:
        raise AssertionError(f"{name}: got {shown}, the lecture says {expected}")


def fetch(url, path, what):
    """Download a file once (quietly), and say in one line what came down."""
    import requests
    if not os.path.exists(path):
        t = time.time()
        with requests.get(url, stream=True, timeout=300) as r:
            r.raise_for_status()
            with open(path + ".part", "wb") as f:
                for chunk in r.iter_content(1 << 22):
                    f.write(chunk)
        os.rename(path + ".part", path)
        print(f"downloaded {what} ({os.path.getsize(path) / 1e6:.1f} MB) in {time.time() - t:.0f} s")
    return path

import gc, hashlib
import pyarrow.parquet as pq
torch.backends.cuda.matmul.allow_tf32 = False      # plain float32 matrix products throughout
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()

SITE = "https://glass-box-sigma.vercel.app/lectures/27-tracing-a-thought/"
def site(name, what, sha256=None):
    """A file from the course site; with sha256, check that it is exactly the file the lecture used."""
    path = fetch(SITE + name, os.path.basename(name), what)
    if sha256:
        assert hashlib.sha256(open(path, "rb").read()).hexdigest() == sha256, f"{name} is not the lecture's file"
    return path

def free():
    gc.collect()
    torch.cuda.empty_cache()

# FineWeb-Edu's first file, read a block of 1,000 documents at a time (the lecture streamed the same file, in order).
from huggingface_hub import HfFileSystem
_fw = pq.ParquetFile(HfFileSystem().open("datasets/HuggingFaceFW/fineweb-edu/sample/10BT/000_00000.parquet", "rb"))
def fineweb():
    for g in range(_fw.num_row_groups):
        yield from _fw.read_row_group(g, columns=["text"]).column("text").to_pylist()

## A question with a hidden middle (Plate I)

**Gemma-2-2B** (Google, 2024) has 26 layers, and each word it reads is held as 2,304 numbers, its **residual stream**.
Every layer adds two things to it: what **attention** brings over from earlier words, and what the **MLP** (a two-layer
network applied to each word on its own) computes. Gemma is gated on Hugging Face, so we use *unsloth/gemma-2-2b*, a
public copy with the same weights, as the lecture did.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
tok = AutoTokenizer.from_pretrained("unsloth/gemma-2-2b")

def load_gemma():
    m = AutoModelForCausalLM.from_pretrained("unsloth/gemma-2-2b", torch_dtype=torch.float32, device_map={"": 0},
                                             attn_implementation="eager").eval()
    for p in m.parameters():
        p.requires_grad_(False)
    return m

t = time.time()
hf = load_gemma()
print(f"Gemma-2-2B: {sum(p.numel() for p in hf.parameters()) / 1e9:.2f} billion numbers, loaded in {time.time() - t:.0f} s")

### Gemma, written out

We only take the library's *weights*. The model itself is short enough to write out, and writing it out is what lets us
freeze parts of it later. Each of the 26 layers does this to the stream `x`:

1. **normalize** `x` (divide by its root-mean-square, then scale each of the 2,304 numbers by a learned weight);
2. **attention**: 8 heads, each scoring every earlier word (with Gemma's "soft cap", which squashes scores beyond ±50),
   turning the scores into a **pattern** of weights that sum to 1, and bringing over a weighted mix of those words'
   *values*. Gemma has only 4 value heads; each serves two of the 8 heads. Normalize the result and add it to `x`.
3. **MLP**: normalize `x` again, run the two-layer network, normalize its output, and add it to `x`.

At the end, one more normalization, then a score for each of the 256,000 words in the vocabulary (squashed beyond ±30).
`rec` records what we need later: the attention patterns, the four normalization scales, the MLP's input and output.

In [ ]:
L, D, H, KV, HD = 26, 2304, 8, 4, 256     # layers, numbers per word, attention heads, value heads, numbers per head
BL = hf.model.layers                      # the 26 layers' weights
W_E = hf.model.embed_tokens.weight        # 256,000 words x 2,304: the word embeddings, also used to score the output

def rms(x):
    return (x.pow(2).mean(-1, keepdim=True) + 1e-6).sqrt()

def rope(T):
    """Rotary position: each pair of numbers in a query or key is turned by an angle that grows with the position."""
    inv = 1.0 / (10000 ** (torch.arange(0, HD, 2, device=DEV).float() / HD))
    a = torch.outer(torch.arange(T, device=DEV).float(), inv)
    a = torch.cat([a, a], -1)
    return a.cos(), a.sin()

def rot(x, cos, sin):
    return x * cos + torch.cat([-x[..., HD // 2:], x[..., :HD // 2]], -1) * sin

def gemma(ids, fz=None, rec=None, mlp=None):
    """Gemma-2-2B on token ids (batch x words); returns the final normalized stream (scores come from logits_of).
    fz:  frozen attention patterns ("P") and, optionally, frozen normalization scales ("s1"-"s4") from an earlier pass.
    rec: a dict that records patterns, scales, each MLP's input u (the stream divided by its RMS) and output mo.
    mlp: mlp(layer, u, mo) may replace a layer's MLP output."""
    fz = fz or {}
    n, T = ids.shape
    x = W_E[ids] * D ** 0.5
    cos, sin = rope(T)
    causal = torch.full((T, T), float("-inf"), device=DEV).triu(1)       # a word sees only itself and earlier words
    if rec is not None:
        rec.update(x0=x, P=[], s1=[], s2=[], s3=[], u=[], mo=[])
    for l in range(L):
        b, a = BL[l], BL[l].self_attn
        s1 = fz["s1"][l] if "s1" in fz else rms(x)
        h = x / s1 * (1 + b.input_layernorm.weight)
        q = rot((h @ a.q_proj.weight.T).view(n, T, H, HD).transpose(1, 2), cos, sin)
        k = rot((h @ a.k_proj.weight.T).view(n, T, KV, HD).transpose(1, 2), cos, sin).repeat_interleave(2, 1)
        v = (h @ a.v_proj.weight.T).view(n, T, KV, HD).transpose(1, 2).repeat_interleave(2, 1)
        if "P" in fz:
            P = fz["P"][l]
        else:
            score = q @ k.transpose(-1, -2) / 16                           # 16 = the square root of 256
            P = (50 * torch.tanh(score / 50) + causal).softmax(-1)          # the pattern: each row sums to 1
        o = (P @ v).transpose(1, 2).reshape(n, T, H * HD) @ a.o_proj.weight.T
        s2 = fz["s2"][l] if "s2" in fz else rms(o)
        x = x + o / s2 * (1 + b.post_attention_layernorm.weight)
        s3 = fz["s3"][l] if "s3" in fz else rms(x)
        u = x / s3                                                          # what a transcoder reads (below)
        m = u * (1 + b.pre_feedforward_layernorm.weight)
        f = b.mlp
        mo = (F.gelu(m @ f.gate_proj.weight.T, approximate="tanh") * (m @ f.up_proj.weight.T)) @ f.down_proj.weight.T
        mo = mo / rms(mo) * (1 + b.post_feedforward_layernorm.weight)
        if mlp is not None:
            mo = mlp(l, u, mo)
        if rec is not None:
            for key, val in (("P", P), ("s1", s1), ("s2", s2), ("s3", s3), ("u", u), ("mo", mo)):
                rec[key].append(val)
        x = x + mo
    s4 = fz["s4"] if "s4" in fz else rms(x)
    if rec is not None:
        rec.update(s4=s4, xL=x)
    return x / s4 * (1 + hf.model.norm.weight)

def logits_of(h):
    return 30 * torch.tanh(h @ W_E.T / 30)

PROMPT = "Fact: the capital of the state containing Dallas is"
ids = tok(PROMPT, return_tensors="pt").input_ids.to(DEV)
T = ids.shape[1]
print([tok.decode(i) for i in ids[0]])
with torch.no_grad():
    ours = logits_of(gemma(ids))[0, -1]
    lib = hf(ids).logits[0, -1]
check("largest gap between our Gemma's scores and the library's", (ours - lib).abs().max().item(), 0, tol=1e-3)

In [ ]:
p = ours.softmax(-1)
pv, pi = p.topk(10)
words = [tok.decode(i) for i in pi]
fig, ax = plt.subplots(figsize=(7.5, 3))
ax.barh(range(10)[::-1], pv.cpu(), color=INK, height=0.6)
ax.set_yticks(range(10)[::-1], [repr(w) for w in words])
ax.set_xlabel("probability of the next word")
ax.set_title("Fact: the capital of the state containing Dallas is …", loc="left", fontsize=11)
for i, v in enumerate(pv.tolist()):
    ax.text(v + 0.005, 9 - i, f"{v:.1%}", va="center", fontsize=9)
plt.show()
check("the top word", words[0], " Austin")
check("P(Austin)", pv[0].item(), 0.41321, tol=0.01)
check("the lecture's top ten, as a set", len(set(words) & set([' Austin', ' not', ' Texas', ' the', ' Fort', ' Dallas', ' Houston', ' ', ' Oklahoma', ' called'])), 10)

The lecture's 41.3% was computed in 16-bit numbers; ours is the same model in 32-bit, a hair lower. The middle words
(*the*, *not*, *Texas*) were tied in the lecture's 16-bit scores (5.6% each) and are untied here.

## Two hops, or one lookup? (Plate II)

Over 35 US cities, ask the two hops separately and then together: *{city} is a city in the state of* (hop 1), *the
capital of {state} is* (hop 2), and the two-hop question. An answer counts as right when Gemma's top word is the
target's first word (*St* for Saint Paul).

In [ ]:
CITIES = [("Dallas", "Texas", "Austin"), ("Houston", "Texas", "Austin"), ("San Antonio", "Texas", "Austin"),
    ("El Paso", "Texas", "Austin"), ("Oakland", "California", "Sacramento"), ("Los Angeles", "California", "Sacramento"),
    ("San Francisco", "California", "Sacramento"), ("San Diego", "California", "Sacramento"),
    ("Miami", "Florida", "Tallahassee"), ("Chicago", "Illinois", "Springfield"), ("Seattle", "Washington", "Olympia"),
    ("Buffalo", "New York", "Albany"), ("Detroit", "Michigan", "Lansing"), ("Philadelphia", "Pennsylvania", "Harrisburg"),
    ("Pittsburgh", "Pennsylvania", "Harrisburg"), ("Las Vegas", "Nevada", "Carson City"), ("Portland", "Oregon", "Salem"),
    ("Milwaukee", "Wisconsin", "Madison"), ("New Orleans", "Louisiana", "Baton Rouge"),
    ("Kansas City", "Missouri", "Jefferson City"), ("Cleveland", "Ohio", "Columbus"), ("Savannah", "Georgia", "Atlanta"),
    ("Baltimore", "Maryland", "Annapolis"), ("Louisville", "Kentucky", "Frankfort"), ("Memphis", "Tennessee", "Nashville"),
    ("Tucson", "Arizona", "Phoenix"), ("Birmingham", "Alabama", "Montgomery"), ("Burlington", "Vermont", "Montpelier"),
    ("Charleston", "South Carolina", "Columbia"), ("Anchorage", "Alaska", "Juneau"), ("Omaha", "Nebraska", "Lincoln"),
    ("Wilmington", "Delaware", "Dover"), ("Newark", "New Jersey", "Trenton"), ("Minneapolis", "Minnesota", "Saint Paul"),
    ("Spokane", "Washington", "Olympia")]

def right(word, target):
    word = word.strip()
    return bool(word) and (word == target.split()[0] or (word == "St" and target.startswith("Saint")))

def top_word(prompt, rec=None):
    with torch.no_grad():
        i = tok(prompt, return_tensors="pt").input_ids.to(DEV)
        return tok.decode(logits_of(gemma(i, rec=rec))[0, -1].argmax())

LECTURE = {'Dallas': [' Texas', ' Austin', ' Austin'], 'Houston': [' Texas', ' Austin', ' Austin'], 'San Antonio': [' Texas', ' Austin', ' Austin'], 'El Paso': [' Texas', ' Austin', ' Austin'], 'Oakland': [' California', ' Sacramento', ' Sacramento'], 'Los Angeles': [' California', ' Sacramento', ' Sacramento'], 'San Francisco': [' California', ' Sacramento', ' Sacramento'], 'San Diego': [' California', ' Sacramento', ' Sacramento'], 'Miami': [' Florida', ' Tallahassee', ' Tallahassee'], 'Chicago': [' Illinois', ' Springfield', ' Springfield'], 'Seattle': [' Washington', ' the', ' Olympia'], 'Buffalo': [' New', ' New', ' Albany'], 'Detroit': [' Michigan', ' Lansing', ' Lansing'], 'Philadelphia': [' Pennsylvania', ' Harrisburg', ' Harrisburg'], 'Pittsburgh': [' Pennsylvania', ' Harrisburg', ' Harrisburg'], 'Las Vegas': [' Nevada', ' Carson', ' Reno'], 'Portland': [' Oregon', ' Salem', ' Salem'], 'Milwaukee': [' Wisconsin', ' Madison', ' Madison'], 'New Orleans': [' Louisiana', ' Baton', ' Baton'], 'Kansas City': [' Missouri', ' Jefferson', ' Topeka'], 'Cleveland': [' Ohio', ' Columbus', ' Columbus'], 'Savannah': [' Georgia', ' Tbilisi', ' Atlanta'], 'Baltimore': [' Maryland', ' Annapolis', ' Annapolis'], 'Louisville': [' Kentucky', ' Frankfort', ' Frankfort'], 'Memphis': [' Tennessee', ' Nashville', ' Nashville'], 'Tucson': [' Arizona', ' Phoenix', ' Phoenix'], 'Birmingham': [' Alabama', ' Montgomery', ' the'], 'Burlington': [' Vermont', ' Mont', ' the'], 'Charleston': [' South', ' Columbia', ' Columbia'], 'Anchorage': [' Alaska', ' Juneau', ' Juneau'], 'Omaha': [' Nebraska', ' Lincoln', ' Lincoln'], 'Wilmington': [' North', ' Dover', ' Raleigh'], 'Newark': [' New', ' Trenton', ' Trenton'], 'Minneapolis': [' Minnesota', ' St', ' St'], 'Spokane': [' Washington', ' the', ' Olympia']}
hops = []
for city, state, cap in CITIES:
    w = [top_word(f"Fact: {city} is a city in the state of"), top_word(f"Fact: the capital of {state} is"),
         top_word(f"Fact: the capital of the state containing {city} is")]
    hops.append([right(w[0], state), right(w[1], cap), right(w[2], cap)])
    for j, name in enumerate(("hop 1", "hop 2", "both hops")):
        if w[j] != LECTURE[city][j]:
            print(f"{city}, {name}: Gemma says {w[j]!r} here, {LECTURE[city][j]!r} in the lecture")
hops = np.array(hops)
fig, ax = plt.subplots(figsize=(11, 2.1))
for j in range(3):
    for i in range(35):
        ax.text(i, 2 - j, "✓" if hops[i, j] else "✗", ha="center", va="center", fontsize=10, color=INK if hops[i, j] else INK2)
ax.set_xlim(-0.6, 34.6); ax.set_ylim(-0.6, 2.6)
ax.set_yticks([2, 1, 0], [f"hop 1  ({hops[:, 0].sum()})", f"hop 2  ({hops[:, 1].sum()})", f"both  ({hops[:, 2].sum()})"])
ax.set_xticks(range(35), [c for c, _, _ in CITIES], rotation=70, fontsize=8)
for s in ax.spines.values():
    s.set_visible(False)
ax.tick_params(length=0)
plt.show()
check("hop 1 right", int(hops[:, 0].sum()), 34)
check("hop 2 right", int(hops[:, 1].sum()), 30, tol=1)
check("both hops at once right", int(hops[:, 2].sum()), 30)

One answer differs from the lecture's: in 32-bit numbers, Gemma names Louisville rather than Frankfort as Kentucky's
capital (hop 2). Everything else matches, city by city.

Gemma knows each hop, and answers the two-hop question about as often as the harder hop alone. That fits two hops, but
it would also fit a memorised lookup. To tell them apart we need to see *inside* (Plate III): the MLPs, where most of
the facts are thought to live, are dense and unreadable, and the prism (Lecture 25) reads the stream but not what an MLP
*does* to it.

---
## Part the First: a readable copy of the MLP

### The transcoder (Plate IV)

A **transcoder** is a prism with a different job. It reads the MLP's *input*, switches on a few of its 16,384 features,
and from them writes a guess of the MLP's *output*. Each feature then means one thing on its way in (when it fires) and
one thing on its way out (what it adds). Gemma Scope released one per layer; the circuit-tracer library uses the
lowest-L0 set (*mwhanna/gemma-scope-transcoders*, the same weights saved for quick loading), 26 files of 302 MB.

Each feature is a **JumpReLU**: its pre-activation is `u · W_enc[f] + b_enc[f]`, and it passes through only above a
learned threshold. `u` is the stream divided by its RMS, *before* the norm's own weights: that is what circuit-tracer
feeds the transcoders, so we do the same.

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from huggingface_hub import hf_hub_download
from safetensors import safe_open

t = time.time()
with ThreadPoolExecutor(4) as pool:
    TC = list(pool.map(lambda l: hf_hub_download("mwhanna/gemma-scope-transcoders", f"layer_{l}.safetensors"), range(L)))
print(f"downloaded 26 transcoders ({sum(os.path.getsize(p) for p in TC) / 1e9:.1f} GB) in {time.time() - t:.0f} s")

def transcoder(l):
    """Layer l's transcoder on the GPU: W_enc and W_dec (16,384 x 2,304), b_enc, b_dec, and the thresholds."""
    with safe_open(TC[l], "pt", device=DEV) as f:
        t = {k: f.get_tensor(k) for k in f.keys()}
    t["thr"] = t.pop("activation_function.threshold")
    return t

def encode(t, u):
    pre = u @ t["W_enc"].T + t["b_enc"]
    return pre * (pre > t["thr"])                 # JumpReLU: the value passes only above the threshold

def feature(l, f):
    """One feature's encoder row, bias, threshold and decoder row, read without loading the whole layer."""
    with safe_open(TC[l], "pt", device=DEV) as h:
        return (h.get_slice("W_enc")[f], h.get_slice("b_enc")[f:f + 1][0],
                h.get_slice("activation_function.threshold")[f:f + 1][0], h.get_slice("W_dec")[f])

In [ ]:
rec = {}
with torch.no_grad():
    gemma(ids, rec=rec)
t20 = transcoder(20)
a20 = encode(t20, rec["u"][20][0])
copy20 = a20 @ t20["W_dec"] + t20["b_dec"]
real20 = rec["mo"][20][0]
on = a20[9].nonzero().squeeze(1)
on = on[a20[9, on].argsort(descending=True)]
fvu20 = ((real20[1:] - copy20[1:]) ** 2).sum() / ((real20[1:] - real20[1:].mean(0)) ** 2).sum()
print(f"layer 20, at 'Dallas': {len(on)} of 16,384 features fire; the copy misses {fvu20:.0%} of the MLP's output "
      f"across the prompt (its variance)")
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.4), gridspec_kw={"width_ratios": [1.2, 1]})
a1.bar(range(len(on)), a20[9, on].cpu(), color=INK)
a1.bar(range(len(on)), t20["thr"][on].cpu(), color=PAPER, edgecolor=INK2, hatch="////", width=0.8)
a1.set_xticks(range(len(on)), [f"20/{f}" for f in on.tolist()], rotation=60, fontsize=8)
a1.set_ylabel("activation"); a1.set_title("features firing at 'Dallas' (hatched: threshold)", loc="left", fontsize=10)
a2.scatter(real20[9].cpu(), copy20[9].cpu(), s=3, color=INK)
lim = real20[9].abs().max().item()
a2.plot([-lim, lim], [-lim, lim], color=INK2, lw=0.8)
a2.set_xlabel("the MLP's output (2,304 numbers)"); a2.set_ylabel("the transcoder's copy")
a2.set_title("layer 20 at 'Dallas'", loc="left", fontsize=10)
plt.tight_layout(); plt.show()
check("features firing at layer 20, 'Dallas'", len(on), 12)
check("20/15589 at 'Dallas' (the lecture's Texas feature)", a20[9, 15589].item(), 46.0, tol=0.15)
check("its threshold", t20["thr"][15589].item(), 5.5257, tol=1e-3)

### One feature, read both ways: Gemma's half (Plate VI)

A transcoder feature's decoder row is a direction it *writes* into the stream. Read it straight through the
unembedding (the scores for the 256,000 words) and you see what it pushes the model to say. Feature **23/12237**, which
the lecture calls *say Austin*, fires on the last word of the prompt.

In [ ]:
def writes(l, f, n=6):
    dl = feature(l, f)[3] @ W_E.T
    return [tok.decode(i) for i in dl.topk(n).indices]

w = writes(23, 12237)
print("23/12237 writes:", w)
check("its top two words", w[:2], ['ModelAdmin', ' Austin'])
del t20, a20, copy20, real20, hf, BL, W_E, ours, lib
free()

Half clean: a code token first, *Austin* second. We now set Gemma aside for a while (it fills 10.5 of the T4's 15 GB)
and build transcoders ourselves, on the smaller GPT-2.

### We train one ourselves (Plate V)

GPT-2 small, block 6 of 12. The lecture trained, on the same 40 million words of FineWeb-Edu, a transcoder (MLP input →
MLP output) and a prism (MLP output → MLP output), each with 12,288 features and **TopK**: only the 32 largest
pre-activations survive for each word, the rest are zeroed. First the text: the lecture's held-out set is the first
2,048 rows of 128 tokens (each row starts with GPT-2's end-of-text token), packed from FineWeb-Edu in order.

In [ ]:
from transformers import GPT2LMHeadModel, GPT2TokenizerFast
gtok = GPT2TokenizerFast.from_pretrained("openai-community/gpt2")
gpt2 = GPT2LMHeadModel.from_pretrained("openai-community/gpt2").to(DEV).eval()
EOS = gtok.eos_token_id
gdocs = fineweb()

def rows(n):
    """The next n rows of 128 tokens: end-of-text, then 127 tokens of documents joined by end-of-text."""
    out, buf = [], []
    while len(out) < n:
        buf.extend(gtok(next(gdocs))["input_ids"] + [EOS])
        while len(buf) >= 127 and len(out) < n:
            out.append([EOS] + buf[:127])
            buf = buf[127:]
    return torch.tensor(out)

held = rows(2048)          # held out: the lecture measured every model on these 260,096 tokens
probe = rows(64)           # used once, to set each stream's scale
print(f"held-out text: {held.shape[0]} rows, {held[:, 1:].numel():,} tokens")

def acts(t, layers=(6,)):
    """MLP input (after ln_2) and output at the given blocks, position 0 dropped: {('in', l): ..., ('out', l): ...}."""
    cap, hooks = {}, []
    for l in layers:
        hooks.append(gpt2.transformer.h[l].mlp.register_forward_hook(
            lambda mod, i, o, l=l: cap.update({("in", l): i[0], ("out", l): o})))
    with torch.no_grad():
        gpt2.transformer(t.to(DEV))
    for h in hooks:
        h.remove()
    return {k: v[:, 1:].reshape(-1, 768).float() for k, v in cap.items()}

class TopK(torch.nn.Module):
    """A TopK transcoder (or prism): x -> keep the k largest of W_enc (x - b_pre) + b_enc -> W_dec z + b_dec."""
    def __init__(self, n, k, d=768, same=False):
        super().__init__()
        self.k = k
        W = torch.randn(n, d) / d ** 0.5
        Wd = W.clone() if same else torch.randn(n, d)
        self.W_enc = torch.nn.Parameter(W)
        self.b_enc = torch.nn.Parameter(torch.zeros(n))
        self.b_pre = torch.nn.Parameter(torch.zeros(d))
        self.W_dec = torch.nn.Parameter(Wd / Wd.norm(dim=1, keepdim=True))     # each decoder row has length 1
        self.b_dec = torch.nn.Parameter(torch.zeros(d))

    def encode(self, x):
        pre = (x - self.b_pre) @ self.W_enc.T + self.b_enc
        v, i = pre.topk(self.k, -1)
        return torch.zeros_like(pre).scatter_(-1, i, torch.relu(v))

    def forward(self, x):
        return self.encode(x) @ self.W_dec + self.b_dec

In [ ]:
def load_topk(name, sha256, what):
    z = np.load(site(f"notebook/{name}", what, sha256))
    m = TopK(*z["W_enc"].shape[:1], int(z["k"])).to(DEV)
    for key in ("W_enc", "W_dec", "b_enc", "b_pre", "b_dec"):
        getattr(m, key).data = torch.tensor(z[key].astype(np.float32), device=DEV)
    return m.requires_grad_(False), float(z["scale_in"]), float(z["scale_out"])

tc6, s_in, s_out = load_topk("gpt2_transcoder_k32.npz", '326c5b22b92de5bab1ee5948b3988c7060bbe2b600b8df590cab046ff1543083', "the lecture's GPT-2 transcoder")
pr6, _, _ = load_topk("gpt2_prism_k32.npz", '305d045d5a27a8dce04bdfa3cebf33f7bbc0dca6533d030081ff69daeb19011c', "the lecture's GPT-2 prism")

res = {}
for name, m in (("transcoder", tc6), ("prism", pr6)):
    sse = sst = 0.0
    fired = torch.zeros(12288, dtype=torch.bool, device=DEV)
    for b0 in range(0, 2048, 256):
        A = acts(held[b0:b0 + 256])
        y = A[("out", 6)] * s_out                                  # the streams are scaled to an average length² of 768
        x = A[("in", 6)] * s_in if name == "transcoder" else y
        for c in range(0, len(y), 8128):
            z = m.encode(x[c:c + 8128])
            sse += ((y[c:c + 8128] - z @ m.W_dec - m.b_dec) ** 2).sum().item()
            fired |= (z > 0).any(0)
        sst += ((y - y.mean(0)) ** 2).sum().item()
    res[name] = (sse / sst, 1 - fired.float().mean().item())
    print(f"{name:10s}: leaves {sse / sst:.1%} of the MLP output unexplained; {1 - fired.float().mean().item():.1%} of "
          f"its features never fire on the held-out text")
check("transcoder: unexplained share", res["transcoder"][0], 0.4284, tol=0.002)
check("prism: unexplained share", res["prism"][0], 0.3471, tol=0.002)
check("transcoder: features that never fire", res["transcoder"][1], 0.3316, tol=0.003)
check("prism: features that never fire", res["prism"][1], 0.0001, tol=0.003)

The transcoder has the harder job: it must *predict* the MLP's output from its input, not copy a vector it can see. At
the same 32 features per word it leaves 23% more unexplained, and a third of its features never fire. The lecture
trained both with an extra term (AuxK) that pushes unused features to fit what the others miss; it keeps the prism's
features alive, but not the transcoder's.

Does it matter for the model? Splice each copy in place of block 6's MLP and measure GPT-2's next-word loss on 512
held-out rows, against two baselines: the real MLP, and the MLP replaced by its average output.

In [ ]:
mu6 = acts(held[:256])[("out", 6)].mean(0)

def next_word_loss(replace=None):
    hook = None
    if replace is not None:
        hook = gpt2.transformer.h[6].mlp.register_forward_hook(lambda mod, i, o: replace(i[0], o))
    tot = n = 0
    with torch.no_grad():
        for b0 in range(0, 512, 32):
            t = held[b0:b0 + 32].to(DEV)
            lg = gpt2(t).logits[:, 1:-1]
            tot += F.cross_entropy(lg.reshape(-1, lg.shape[-1]), t[:, 2:].reshape(-1), reduction="sum").item()
            n += t[:, 2:].numel()
    if hook:
        hook.remove()
    return tot / n

ce = {"clean": next_word_loss(), "average": next_word_loss(lambda i, o: mu6.expand_as(o)),
      "transcoder": next_word_loss(lambda i, o: tc6(i.float() * s_in) / s_out),
      "prism": next_word_loss(lambda i, o: pr6(o.float() * s_out) / s_out)}
for k in ("transcoder", "prism"):
    rec_ = (ce["average"] - ce[k]) / (ce["average"] - ce["clean"])
    print(f"{k:10s}: loss {ce[k]:.4f} (clean {ce['clean']:.4f}, MLP averaged {ce['average']:.4f}): "
          f"{rec_:.1%} of the loss recovered")
    check(f"{k}: loss recovered", rec_, {'transcoder': 0.5919, 'prism': 0.7295}[k], tol=0.02)
check("clean next-word loss", ce["clean"], 3.6842, tol=0.003)

Block 6's MLP matters only 0.06 nats here, so these percentages sit on a small difference; the order is what counts.

### One feature, read both ways: GPT-2's half (Plate VI)

The *input side* of a feature is where it fires: its top contexts on the held-out text. The *output side* is what it
writes, read through GPT-2's unembedding. The lecture showed the first 24 features that fire on between 0.03% and 0.3%
of tokens.

In [ ]:
freq = torch.zeros(12288, device=DEV)
best_v = torch.zeros(12288, 8, device=DEV)
best_i = torch.zeros(12288, 8, dtype=torch.long, device=DEV)
for b0 in range(0, 2048, 256):
    x = acts(held[b0:b0 + 256])[("in", 6)] * s_in
    for c in range(0, len(x), 8128):
        z = tc6.encode(x[c:c + 8128])
        freq += (z > 0).float().sum(0)
        v, i = z.T.topk(8, dim=1)
        cv, ci = torch.cat([best_v, v], 1), torch.cat([best_i, i + b0 * 127 + c], 1)
        top = cv.topk(8, dim=1).indices
        best_v, best_i = cv.gather(1, top), ci.gather(1, top)
freq /= held[:, 1:].numel()
shown = ((freq > 3e-4) & (freq < 3e-3)).nonzero().squeeze(1)[:24].tolist()

def contexts(f, n=4):
    out = []
    for v, j in zip(best_v[f].tolist()[:n], best_i[f].tolist()[:n]):
        r, p = divmod(j, 127)
        row = held[r].tolist()
        out.append(f"{gtok.decode(row[max(1, p + 1 - 8):p + 1])}[{gtok.decode(row[p + 1])}]{gtok.decode(row[p + 2:p + 4])}")
    return out

W_U = gpt2.lm_head.weight
for f in shown[:3]:
    print(f"feature {f} (fires on {freq[f]:.2%} of tokens)")
    for c in contexts(f):
        print("   ", repr(c))
    print("    writes:", [gtok.decode(i) for i in (tc6.W_dec[f] / s_out @ W_U.T).topk(6).indices])
check("the 24 features shown", shown, [0, 2, 5, 6, 14, 15, 19, 23, 36, 37, 39, 44, 46, 50, 51, 53, 55, 57, 58, 59, 64, 68, 70, 71])
j0 = best_i[0, 0].item()
check("feature 0's strongest context word", gtok.decode(held[j0 // 127, j0 % 127 + 1].item()), ' believe')

Feature 0 fires on *believe*, feature 2 on *offer* after *can* or *may*: readable on the way in. Their output sides are
not clean words. Block 6 of 12 writes directions that later blocks still transform, while Gemma's layer-23 feature sat
three layers from the output and read as *Austin*.

### Cross-layer transcoders (Plate VII)

A per-layer transcoder must rebuild each MLP from scratch. A **cross-layer transcoder** lets a feature read at one block
and write to that block's MLP output *and every later one's*. Here are both, by hand, for blocks 0–3: four per-layer
transcoders, and one cross-layer transcoder with the same 3,072 features per block and 16 on per word, trained side by
side on the same text with the lecture's recipe (Adam, learning rate 2 × 10⁻⁴, 4,096 tokens per step). We run 250 steps,
the first point of the lecture's training log (it ran 9,772), with the big products in 16-bit numbers to fit the free
GPU's time.

In [ ]:
torch.manual_seed(0)
P4 = acts(probe, layers=(0, 1, 2, 3))
sc = {k: (768 / (v ** 2).sum(-1).mean()).sqrt().item() for k, v in P4.items()}   # scale each stream to length² 768
per_layer = [TopK(3072, 16).to(DEV) for _ in range(4)]
clt_enc = [TopK(3072, 16).to(DEV) for _ in range(4)]                       # (their own decoders go unused)
clt_dec = {(s, t): torch.nn.Parameter(torch.randn(3072, 768, device=DEV) / 768 ** 0.5 / (t - s + 1))
           for s in range(4) for t in range(s, 4)}                         # block s's features -> block t's output
clt_b = torch.nn.Parameter(torch.zeros(4, 768, device=DEV))
with torch.no_grad():
    for l in range(4):
        per_layer[l].b_pre.data = clt_enc[l].b_pre.data = P4[("in", l)].mul(sc[("in", l)]).mean(0)

def clt(xs):
    z = [clt_enc[s].encode(xs[s]) for s in range(4)]
    return [clt_b[t] + sum(z[s] @ clt_dec[(s, t)] for s in range(t + 1)) for t in range(4)]

def fvu(y, yh):
    return ((y - yh) ** 2).sum(-1).mean() / ((y - y.mean(0)) ** 2).sum(-1).mean()

params = [p for m in per_layer + clt_enc for p in m.parameters()] + list(clt_dec.values()) + [clt_b]
opt = torch.optim.Adam(params, lr=2e-4)
scaler = torch.amp.GradScaler("cuda")
t, step = time.time(), 0
while step < 250:
    A = acts(rows(256), layers=(0, 1, 2, 3))
    perm = torch.randperm(len(A[("in", 0)]), device=DEV)
    for b0 in range(0, len(perm) - 4095, 4096):
        if step == 250:
            break
        sl = perm[b0:b0 + 4096]
        xs = [A[("in", l)][sl] * sc[("in", l)] for l in range(4)]
        ys = [A[("out", l)][sl] * sc[("out", l)] for l in range(4)]
        with torch.autocast("cuda", dtype=torch.float16):
            f_pl = [fvu(ys[l], per_layer[l](xs[l]).float()) for l in range(4)]
            f_cl = [fvu(ys[l], yh.float()) for l, yh in enumerate(clt(xs))]
            loss = sum(f_pl) + sum(f_cl) / 4
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        with torch.no_grad():
            for m in per_layer:
                m.W_dec.data /= m.W_dec.data.norm(dim=1, keepdim=True)
        step += 1
print(f"250 steps ({250 * 4096:,} tokens) in {time.time() - t:.0f} s; on the last batch:")
print("  per-layer, blocks 0-3:", [round(f.item(), 3) for f in f_pl], "  cross-layer, average:", round(sum(f_cl).item() / 4, 3))
for l in range(4):
    check(f"per-layer transcoder, block {l}, at step 250", f_pl[l].item(), {'plt4_l0_k16': 0.6382, 'plt4_l1_k16': 0.8003, 'plt4_l2_k16': 0.8975, 'plt4_l3_k16': 0.9323, 'clt_mean_fvu': 0.7447}[f"plt4_l{l}_k16"], tol=0.02)
check("cross-layer transcoder, average over blocks, at step 250", sum(f_cl).item() / 4, {'plt4_l0_k16': 0.6382, 'plt4_l1_k16': 0.8003, 'plt4_l2_k16': 0.8975, 'plt4_l3_k16': 0.9323, 'clt_mean_fvu': 0.7447}["clt_mean_fvu"], tol=0.02)

In [ ]:
sse = torch.zeros(2, 4)
sst = torch.zeros(4)
with torch.no_grad():
    for b0 in range(0, 2048, 256):
        A = acts(held[b0:b0 + 256], layers=(0, 1, 2, 3))
        xs = [A[("in", l)] * sc[("in", l)] for l in range(4)]
        ys = [A[("out", l)] * sc[("out", l)] for l in range(4)]
        cl = clt(xs)
        for l in range(4):
            sse[0, l] += ((ys[l] - per_layer[l](xs[l])) ** 2).sum().item()
            sse[1, l] += ((ys[l] - cl[l]) ** 2).sum().item()
            sst[l] += ((ys[l] - ys[l].mean(0)) ** 2).sum().item()
ours4 = sse / sst
lect = np.array([[0.203, 0.4071, 0.476, 0.5096], [0.2094, 0.3484, 0.3687, 0.3827]])
fig, ax = plt.subplots(figsize=(7.5, 3.2))
for j, (lab, ls) in enumerate((("per-layer", "-"), ("cross-layer", "--"))):
    ax.plot(range(4), ours4[j], ls, color=INK, marker="o", label=f"{lab}, ours (250 steps)")
    ax.plot(range(4), lect[j], ls, color=INK2, marker="s", label=f"{lab}, the lecture's (9,772 steps)")
ax.set_xticks(range(4), [f"block {l}" for l in range(4)])
ax.set_ylabel("share of the MLP output unexplained"); ax.set_ylim(0, 1)
ax.legend(fontsize=8, ncol=2); plt.show()
for l in range(4):
    print(f"block {l}: per-layer {ours4[0, l]:.3f}, cross-layer {ours4[1, l]:.3f}   "
          f"(lecture, fully trained: {lect[0, l]:.3f}, {lect[1, l]:.3f})")
check("cross-layer better than per-layer on blocks 1-3", bool((ours4[1, 1:] < ours4[0, 1:]).all()), True)

At block 0 there is nothing earlier to borrow from, so the two tie (in the lecture; after only 250 steps ours have not
settled there yet). From block 1 on, earlier features write forward and the cross-layer copy leaves less unexplained.
The lecture's graphs below use Gemma's per-layer transcoders: Gemma's cross-layer transcoder (426,000 features, each
writing to every later layer) is about 26 GB, too big for a free GPU.

---
## Part the Second: the network as a sum of arrows

### The replacement model (Plate VIII)

Back to Gemma. Run the prompt once, and at every layer and word let the transcoder read the MLP's input. Three things
come out per layer: which features fire (and how strongly), the transcoder's copy, and the **error**: the real output
minus the copy. Like circuit-tracer we skip the first token, `<bos>`, where transcoders were not trained.

In [ ]:
for name in ("gpt2", "tc6", "pr6", "per_layer", "clt_enc", "clt_dec", "clt_b", "opt", "params", "scaler", "A", "P4",
             "held", "x", "y", "z", "xs", "ys", "cl", "loss", "f_pl", "f_cl"):
    globals().pop(name, None)                  # GPT-2's part is done: give the GPU back to Gemma
free()
hf = load_gemma()
BL, W_E = hf.model.layers, hf.model.embed_tokens.weight

rec = {}
with torch.no_grad():
    hL = gemma(ids, rec=rec)
feats, enc_rows, dec_rows, err, bdec = [], [], [], [], []
t = time.time()
with torch.no_grad():
    for l in range(L):
        tl = transcoder(l)
        a = encode(tl, rec["u"][l][0])
        a[0] = 0
        e = rec["mo"][l][0] - (a @ tl["W_dec"] + tl["b_dec"])     # the error node: what the copy misses
        e[0] = 0
        pp, ff = a.nonzero(as_tuple=True)
        feats += [(l, p, f, a[p, f].item()) for p, f in zip(pp.tolist(), ff.tolist())]
        enc_rows.append(tl["W_enc"][ff])
        dec_rows.append(tl["W_dec"][ff] * a[pp, ff][:, None])     # each active feature's write: activation x decoder row
        err.append(e)
        bdec.append(tl["b_dec"])
        del tl
enc_rows, dec_rows, err = torch.cat(enc_rows), torch.cat(dec_rows), torch.stack(err)
NF = len(feats)
F_l = torch.tensor([f[0] for f in feats], device=DEV)
F_p = torch.tensor([f[1] for f in feats], device=DEV)
ACT = torch.tensor([f[3] for f in feats], device=DEV)
idx = {(l, p, f): i for i, (l, p, f, _) in enumerate(feats)}
print(f"{NF:,} features active, of {L * T * 16384:,} possible ({NF / (L * T * 16384):.2%}); {L * T} error nodes; "
      f"{time.time() - t:.0f} s")

def rebuilt(l, u, mo):        # the MLP output, rebuilt from its parts: features + decoder bias + error
    out = torch.zeros_like(mo)
    m = F_l == l
    out[0].index_add_(0, F_p[m], dec_rows[m])
    out[0, 1:] += bdec[l] + err[l][1:]
    out[0, 0] = rec["mo"][l][0, 0]
    return out
with torch.no_grad():
    same = logits_of(gemma(ids, mlp=rebuilt))[0, -1]
check("largest change in any score, with every MLP rebuilt from features + errors", (same - logits_of(hL)[0, -1]).abs().max().item(), 0, tol=1e-3)
check("active features", NF, 9161, tol=30)

With the error nodes kept, the copy is exact by construction. The lecture's count of active features (9,161) was made in
16-bit numbers; a few dozen features sit right at their thresholds and tip one way or the other.

### How good is the copy on its own? (Plate IX)

Now drop the error nodes: replace every MLP by its transcoder alone and let the model run. The lecture tested this on 64
held-out FineWeb-Edu documents (the last 64 of the first 6,000 that have at least 128 tokens), comparing the copy's top
next word with real Gemma's at 8,064 positions. We rebuild the same 64 documents.

In [ ]:
docs = []
for text in fineweb():
    i = tok(text[:4000])["input_ids"]                  # 4,000 characters is far more than 128 tokens
    if len(i) >= 128:
        docs.append(i[:128])
    if len(docs) == 6000:
        break
test = torch.tensor(docs[-64:], device=DEV)
check("the lecture's six passages, token for token", [[tok.decode(i) for i in r[:48]] for r in test[:6].tolist()],
      [['<bos>', 'Hom', 'onyms', ' are', ' words', ' that', ' have', ' the', ' same', ' spelling', ' but', ' different', ' meanings', '.', ' “', 'I', '’', 'm', ' sorry', '”', ' is', ' what', ' I', ' dub', ' a', ' hom', 'onym', ' phrase', ' because', ' there', ' are', ',', ' based', ' on', ' my', ' observations', ' of', ' the', ' world', ',', ' four', ' major', ' ways', ' this', ' phrase', ' is', ' used', '.'], ['<bos>', 'Your', ' Challenge', ':', ' Can', ' you', ' build', ' a', ' bristle', ' bot', ' and', ' make', ' it', ' move', '?', '\n', 'What', ' You', ' Need', '\n', '-', ' ', '1', ' motor', ' with', ' two', ' wires', ' (', '1', '-', 'volt', ' to', ' ', '3', '-', 'volt', ' motor', ')', ' You', ' can', ' buy', ' this', ' at', ' an', ' electronics', ' store', ' or', ' remove'], ['<bos>', 'The', ' influence', ' of', ' money', ' in', ' politics', '\n', '7', '6', '%', ' of', ' Americans', ' think', ' that', ' the', ' amount', ' of', ' money', ' in', ' elections', ' gives', ' rich', ' people', ' more', ' influence', ' than', ' the', ' rest', ' of', ' us', '.', ' They', "'", 're', ' right', '.', ' Princeton', ' and', ' Northwestern', ' University', ' recently', ' released', ' a', ' released', ' a', ' study', ' suggesting'], ['<bos>', 'While', ' we', ' celebrate', ' Rosh', ' Hash', 'anah', ' as', ' the', ' Jewish', ' New', ' Year', ',', ' as', ' the', ' day', ' the', ' Hebrew', ' calendar', ' begins', ',', ' that', ' wasn', '’', 't', ' always', ' the', ' case', '.', ' Nowadays', ' we', ' celebrate', ' Rosh', ' Hash', 'anah', ' on', ' the', ' first', ' day', ' of', ' the', ' fall', ' month', ' of', ' T', 'ish', 'rei', '.'], ['<bos>', 'Published', ':', ' ', '1', '0', ' October', ' ', '2', '0', '1', '4', ' at', ' ', '1', '3', ':', '5', '3', '\n', 'Ang', 'lia', ' Ruskin', ' scientist', ' co', '-', 'authors', ' paper', ' showing', ' har', 'le', 'quins', ' prefer', ' urban', ' areas', '\n', 'A', ' new', ' paper', ',', ' co', '-', 'authored', ' by', ' Anglia', ' Ruskin', ' University', ' scientist'], ['<bos>', 'This', ' section', ' presents', ' articles', ' that', ' will', ' help', ' the', ' true', ' truth', ' seeker', ' to', ' understand', ' the', ' religion', ' of', ' Al', '-', 'Islam', '.', ' Articles', ' discussing', ' the', ' basic', ' tenets', ' of', ' the', ' religion', ',', ' religious', ' philosophy', ',', ' and', ' Islamic', ' history', ' as', ' well', ' as', ' articles', ' disp', 'elling', ' common', ' myths', ',', ' propaganda', ' and', ' deliberate']])

_tc = {}
def swap(layers):
    """Replace the MLPs of `layers` by their transcoders, with no error terms (the real output is kept at <bos>)."""
    def h(l, u, mo):
        if l not in layers:
            return mo
        if l not in _tc:
            _tc.clear()
            _tc[l] = transcoder(l)
        t, out = _tc[l], torch.empty_like(mo)
        flat, o = u.reshape(-1, D), out.view(-1, D)
        for c in range(0, len(flat), 2048):
            o[c:c + 2048] = encode(t, flat[c:c + 2048]) @ t["W_dec"] + t["b_dec"]
        out[:, 0] = mo[:, 0]
        return out
    return h

def compare(h_real, h_copy, toks):
    """Top-1 agreement, KL (real || copy), next-word loss and accuracy, positions 1-126."""
    s = dict(agree=0.0, kl=0.0, ce_r=0.0, ce_c=0.0, acc_r=0.0, acc_c=0.0)
    n = 0
    for d in range(len(toks)):
        lr, lc = logits_of(h_real[d:d + 1])[0, 1:-1], logits_of(h_copy[d:d + 1])[0, 1:-1]
        y = toks[d, 2:]
        pr, pc = F.log_softmax(lr, -1), F.log_softmax(lc, -1)
        s["agree"] += (lr.argmax(-1) == lc.argmax(-1)).sum().item()
        s["kl"] += (pr.exp() * (pr - pc)).sum().item()
        s["ce_r"] -= pr.gather(-1, y[:, None]).sum().item(); s["ce_c"] -= pc.gather(-1, y[:, None]).sum().item()
        s["acc_r"] += (lr.argmax(-1) == y).sum().item(); s["acc_c"] += (lc.argmax(-1) == y).sum().item()
        n += len(y)
    return {k: v / n for k, v in s.items()}, n

t = time.time()
with torch.no_grad():
    h_real = gemma(test)
    r26, n = compare(h_real, gemma(test, mlp=swap(set(range(L)))), test)
print(f"all 26 MLPs swapped, {n:,} positions: the copy agrees with Gemma's top word {r26['agree']:.1%} of the time; "
      f"next-word loss {r26['ce_r']:.3f} -> {r26['ce_c']:.3f} nats, accuracy {r26['acc_r']:.1%} -> {r26['acc_c']:.1%} "
      f"({time.time() - t:.0f} s)")
check("top-1 agreement, all 26 MLPs swapped", r26["agree"], {'top1_agreement': 0.3451, 'kl_nats': 2.7065, 'ce_real': 2.5436, 'ce_replacement': 5.1989, 'next_token_acc_real': 0.4681, 'next_token_acc_replacement': 0.2219, 'n_positions': 8064}["top1_agreement"], tol=0.005)
check("KL, nats per word", r26["kl"], {'top1_agreement': 0.3451, 'kl_nats': 2.7065, 'ce_real': 2.5436, 'ce_replacement': 5.1989, 'next_token_acc_real': 0.4681, 'next_token_acc_replacement': 0.2219, 'n_positions': 8064}["kl_nats"], tol=0.05)
check("next-word loss with the copy", r26["ce_c"], {'top1_agreement': 0.3451, 'kl_nats': 2.7065, 'ce_real': 2.5436, 'ce_replacement': 5.1989, 'next_token_acc_real': 0.4681, 'next_token_acc_replacement': 0.2219, 'n_positions': 8064}["ce_replacement"], tol=0.05)
check("accuracy with the copy", r26["acc_c"], {'top1_agreement': 0.3451, 'kl_nats': 2.7065, 'ce_real': 2.5436, 'ce_replacement': 5.1989, 'next_token_acc_real': 0.4681, 'next_token_acc_replacement': 0.2219, 'n_positions': 8064}["next_token_acc_replacement"], tol=0.005)

A third of the time. Each MLP alone is copied well (swap a single layer and agreement stays at 90–95%), but the errors
compound as more layers are swapped. The plate shows the fall on six passages: the first 48 tokens of six of these
documents, with the first *k* MLPs swapped.

In [ ]:
KS = [1, 2, 4, 8, 13, 20, 26]
pas = test[:6, :48]
with torch.no_grad():
    real = logits_of(gemma(pas)).argmax(-1)
    agree = {k: (logits_of(gemma(pas, mlp=swap(set(range(k))))).argmax(-1)[:, 1:] == real[:, 1:]).float().mean(1).cpu()
             for k in KS}
lect = {'1': [0.9787, 0.9362, 0.9787, 0.9574, 0.9149, 0.9574], '2': [0.9574, 0.9149, 0.9574, 0.9574, 0.8936, 0.9787], '4': [0.7872, 0.9149, 0.8723, 0.8936, 0.7872, 0.9574], '8': [0.7021, 0.7447, 0.7447, 0.8298, 0.6596, 0.8511], '13': [0.617, 0.7021, 0.617, 0.7447, 0.617, 0.766], '20': [0.4043, 0.5532, 0.5319, 0.5957, 0.5957, 0.4468], '26': [0.383, 0.234, 0.3404, 0.4043, 0.4255, 0.383]}
fig, ax = plt.subplots(figsize=(7.5, 3.2))
for d in range(6):
    ax.plot(KS, [agree[k][d] for k in KS], color=INK2, lw=0.7, alpha=0.7)
ax.plot(KS, [agree[k].mean() for k in KS], color=INK, lw=2, marker="o", label="ours, average of six passages")
ax.plot(KS, [np.mean(lect[str(k)]) for k in KS], color=INK, lw=1, ls="--", marker="s", label="the lecture's")
ax.set_xlabel("MLPs swapped (the first k layers)"); ax.set_ylabel("agreement with Gemma's top word")
ax.set_ylim(0, 1); ax.legend(fontsize=9); plt.show()
for k in KS:
    check(f"first {k} swapped: average agreement over the six passages", agree[k].mean().item(), np.mean(lect[str(k)]), tol=0.03)

### An edge is a number (Plate X)

Now the main idea. Take the prompt's single run and **freeze** three things at the values that run produced: the
attention patterns, the normalization scales, and every MLP output (each now a fixed sum of feature writes, a decoder
bias and an error node). What is left is a **linear** network: residual additions, attention moving fixed mixtures of
values, and fixed rescalings. In a linear network, any number inside is exactly a sum of contributions, one from each
source.

A feature's **pre-activation** reads the stream along its encoder row. Send that row *backwards* through the frozen
network, and you get, at every earlier layer and word, the direction that source would have to write to move the
target by 1. That is a gradient, and we compute it by hand: `attn_back` is attention run backwards, through the values
only, since the patterns are frozen.

In [ ]:
groups = [[(p, ((F_l == l) & (F_p == p)).nonzero().squeeze(1)) for p in range(T) if ((F_l == l) & (F_p == p)).any()]
          for l in range(L)]

def attn_back(l, G):
    """Layer l's attention with frozen patterns and scales, run backwards: a gradient on the stream after it
    -> the extra gradient it sends to the stream before it."""
    b, a, n = BL[l], BL[l].self_attn, G.shape[0]
    g = G * (1 + b.post_attention_layernorm.weight) / rec["s2"][l]
    g = (g @ a.o_proj.weight).view(n, T, H, HD).transpose(1, 2)
    g = rec["P"][l].transpose(-1, -2) @ g                      # back along the pattern: to the words that were read
    g = g.view(n, KV, 2, T, HD).sum(2)                         # two heads share each value head
    g = g.transpose(1, 2).reshape(n, T, KV * HD) @ a.v_proj.weight
    return g * (1 + b.input_layernorm.weight) / rec["s1"][l]

def direct_effects(t_layer, t_pos, t_vec):
    """For a batch of targets, each reading the stream at (layer, word) along t_vec (layer L = a word's score), the
    direct effect of every source: [features | error nodes, layer by layer | word embeddings], and of the fixed offsets."""
    n = len(t_layer)
    G = torch.zeros(n, T, D, device=DEV)
    top = (t_layer == L).nonzero().squeeze(1)
    G[top, T - 1] = t_vec[top] * (1 + hf.model.norm.weight) / rec["s4"][0, T - 1]
    R = torch.zeros(n, NF + L * T + T, device=DEV)
    offsets = torch.zeros(n, device=DEV)
    for l in reversed(range(L)):
        # G is now the gradient on layer l's MLP output: each source there contributes (its write) . G
        for p, ix in groups[l]:
            R[:, ix] = G[:, p] @ dec_rows[ix].T
        R[:, NF + l * T: NF + (l + 1) * T] = (G * err[l]).sum(-1)
        offsets += G[:, 0] @ rec["mo"][l][0, 0] + G[:, 1:].sum(1) @ bdec[l]   # <bos>'s MLP output, decoder biases
        here = (t_layer == l).nonzero().squeeze(1)
        G[here, t_pos[here]] += t_vec[here] / rec["s3"][l][0, t_pos[here]]   # targets that read this layer's MLP input
        G = G + attn_back(l, G)
    R[:, NF + L * T:] = (G * rec["x0"]).sum(-1)
    return R, offsets

In [ ]:
TEXAS = [(20, 9, 15589), (19, 9, 7477), (16, 9, 25), (4, 9, 13154), (14, 9, 2268), (7, 9, 6861)]
SA = idx[(23, 10, 12237)]                                     # "say Austin", at "is"
with torch.no_grad():
    row, off = direct_effects(F_l[[SA]], F_p[[SA]], enc_rows[[SA]])
row = row[0]
tx = idx[TEXAS[0]]
print(f"edge 20/15589 at 'Dallas' -> 23/12237 at 'is': {row[tx]:.3f} = activation {ACT[tx]:.2f} x virtual weight "
      f"{row[tx] / ACT[tx]:.4f}")
six = [row[idx[x]].item() for x in TEXAS]
print("the six Texas features' edges into 'say Austin':", [round(v, 3) for v in six], f"sum {sum(six):.2f}")
check("the edge", row[tx].item(), 1.711, tol=0.02)
check("its virtual weight", (row[tx] / ACT[tx]).item(), round(1.711 / 46.0, 4), tol=0.0005)
check("the six Texas edges, summed", sum(six), sum([1.711, 1.07, 3.062, 1.477, 2.719, 0.637]), tol=0.1)

# Test it: take the Texas feature's write out of layer 20's MLP output, rerun the frozen network, read the target.
fz = {k: rec[k] for k in ("P", "s1", "s2", "s3", "s4")}
sa_vec = feature(23, 12237)
def pre_sa(minus=None):
    def mlp(l, u, mo):
        out = rec["mo"][l].clone()
        if minus is not None and l == minus[0]:
            out[0, minus[1]] -= dec_rows[minus[2]]
        return out
    r = {}
    with torch.no_grad():
        gemma(ids, fz=fz, rec=r, mlp=mlp)
    return (r["u"][23][0, 10] @ sa_vec[0] + sa_vec[1]).item()
drop = pre_sa() - pre_sa(minus=(20, 9, tx))
print(f"remove the write in the frozen model: say Austin's pre-activation falls by {drop:.4f}")
check("the fall equals the edge", drop - row[tx].item(), 0, tol=2e-3)

An edge is the source's activation times a **virtual weight**: how much one unit of that feature, written at layer 20
over *Dallas*, moves the target feature's input at layer 23 over *is*, through the three layers of frozen attention
between them. And it is not an estimate: remove the write, and the target moves by exactly the edge.

### A feature is the sum of its arrows (Plate XI)

Every source contributes something to *say Austin*. Add them all, then the feature's own bias and the fixed offsets
(the decoder biases and `<bos>`'s MLP output, which are not nodes), and you get its pre-activation exactly. Above its
threshold, a JumpReLU's activation *is* its pre-activation.

In [ ]:
pre = (rec["u"][23][0, 10] @ sa_vec[0] + sa_vec[1]).item()
nz = (row != 0).sum().item()
print(f"{nz:,} sources reach 'say Austin'; they sum to {row.sum():.2f}")
print(f"{row.sum():.2f} (sources) {minus(f'{sa_vec[1].item():+.2f}')} (its bias) {minus(f'{off.item():+.2f}')} (offsets) "
      f"= {row.sum().item() + sa_vec[1].item() + off.item():.3f};  its pre-activation: {pre:.3f}")
check("sources + bias + offsets = pre-activation", row.sum().item() + sa_vec[1].item() + off.item() - pre, 0, tol=2e-3)
check("say Austin's activation", ACT[SA].item(), 53.75, tol=0.25)
check("the sum of its inputs", row.sum().item(), 85.246, tol=0.4)
check("its bias", sa_vec[1].item(), -16.2813, tol=1e-3)
check("the offsets", off.item(), round(53.75 - 85.246 - -16.2813, 3), tol=0.4)

### The whole graph, and pruning it (Plate XII)

Do that for *every* active feature and for the top word scores, and you have the **attribution graph**: one row of
direct effects per target. Logits are scored the circuit-tracer way, along the word's unembedding row minus the average
row, before the final squash, for the top words that make up 95% of the probability (at most 10). Circuit-tracer then
keeps at most 8,192 features: it ranks them by their influence on the output so far, adds the next 1,024, and ranks
again. We follow the same rule.

In [ ]:
t = time.time()
with torch.no_grad():
    lg = logits_of(hL)[0, -1]
    lp, lw = lg.softmax(-1).topk(10)
    keep = int(torch.searchsorted(lp.cumsum(0), 0.95)) + 1
    lp, lw = lp[:keep], lw[:keep]
    NL = len(lw)
    U = W_E[lw] - W_E.mean(0)                                     # demeaned unembedding rows
    RL, _ = direct_effects(torch.full((NL,), L, device=DEV), torch.full((NL,), T - 1, device=DEV), U)
    RF = torch.cat([direct_effects(F_l[i:i + 256], F_p[i:i + 256], enc_rows[i:i + 256])[0] for i in range(0, NF, 256)])
print(f"{NF + NL:,} rows of {NF + L * T + T:,} direct effects ({(NF + NL) * (NF + L * T + T) / 1e6:.0f} million numbers) "
      f"in {time.time() - t:.0f} s")
print("logit targets:", [f"{tok.decode(w)} {p:.1%}" for w, p in zip(lw.tolist(), lp.tolist())])

# choose 8,192 features as circuit-tracer does
N_ALL = NF + (L + 1) * T + NL
def influence(M, w, iters=1000):
    """Total influence on the output: w M + w M² + ... (the graph has no loops, so this ends)."""
    cur = w @ M
    tot = cur.clone()
    for _ in range(iters):
        cur = cur @ M
        if not cur.any():
            break
        tot += cur
    return tot

def normed(M):
    M = M.abs()
    return M / M.sum(1, keepdim=True).clamp(min=1e-10)

E_ = torch.zeros(NL + NF, N_ALL, device=DEV)
E_[:NL, :NF + L * T + T] = RL
who = torch.zeros(NL + NF, dtype=torch.long, device=DEV)
who[:NL] = torch.arange(NL, device=DEV) + NF + (L + 1) * T
seen = torch.zeros(NF, dtype=torch.bool, device=DEV)
st = NL
while st - NL < 8192 and st - NL < NF:
    w0 = torch.zeros(N_ALL, device=DEV); w0[-NL:] = lp
    cur, inf_ = w0, torch.zeros(N_ALL, device=DEV)
    Mn = normed(E_[:st])
    for _ in range(128):
        cur = cur[who[:st]] @ Mn
        if not cur.any():
            break
        inf_ += cur
    order = inf_[:NF].argsort(descending=True)
    nxt = order[~seen[order]][:min(1024, 8192 - (st - NL))]
    E_[st:st + len(nxt), :NF + L * T + T] = RF[nxt]
    who[st:st + len(nxt)] = nxt
    seen[nxt] = True
    st += len(nxt)
sel = seen.nonzero().squeeze(1)
NS = len(sel)
cols = torch.cat([sel, torch.arange(NF, N_ALL, device=DEV)])
rows_ = E_[:st][:, cols][who[:st].argsort()]
A = torch.zeros(len(cols), len(cols), device=DEV)            # the graph: A[target, source]
A[:NS] = rows_[:NS]
A[-NL:] = rows_[NS:]
del E_, rows_, Mn, RF
free()
col = {feats[i][:3]: j for j, i in enumerate(sel.tolist())}  # (layer, word, feature) -> column of A
e0, e1 = NS, NS + L * T                                       # error nodes, then the 11 word embeddings, then logits
w = torch.zeros(len(cols), device=DEV); w[-NL:] = lp
print(f"kept {NS:,} of {NF:,} features")
check("features kept", NS, 8192)

In [ ]:
def threshold(s, frac):
    s = s[s > 0].sort(descending=True).values
    i = int(torch.searchsorted(s.cumsum(0) / s.sum(), frac))
    return s[min(i, len(s) - 1)]

def prune(node_frac, edge_frac=0.98):
    """circuit-tracer's prune_graph: keep the nodes carrying node_frac of the influence, then the edges carrying
    edge_frac, then drop nodes left with no way in or out."""
    ni = influence(normed(A), w)
    nm = ni >= threshold(ni, node_frac)
    nm[-NL - T:] = True                                        # always keep the words and the logits
    Ap = A.clone(); Ap[~nm] = 0; Ap[:, ~nm] = 0
    Np = normed(Ap); del Ap
    es = Np * (influence(Np, w) + w)[:, None]; del Np
    em = es >= threshold(es.flatten(), edge_frac); del es
    old = None
    while old is None or not torch.equal(nm, old):
        old = nm.clone()
        em[~nm] = False; em[:, ~nm] = False
        nm[:-NL - T] &= em[:, :-NL - T].any(0)               # a feature or error node must send an edge
        nm[:NS] &= em[:NS].any(1)                             # a feature must receive one
    return nm, em

def scores(M):
    """Replacement score: the share of word-to-output influence that avoids error nodes.
    Completeness: the influence-weighted share of every node's input that does not come from error nodes."""
    Mn = normed(M)
    inf_ = influence(Mn, w)
    tokens, errors = inf_[e1:e1 + T].sum(), inf_[e0:e1].sum()
    o = inf_ + w
    return (tokens / (tokens + errors)).item(), ((1 - Mn[:, e0:e1].sum(-1)) * o).sum().item() / o.sum().item()

rep, comp = scores(A)
print(f"unpruned: replacement score {rep:.3f}, completeness {comp:.3f}")
sweep, level = [], torch.full((NS,), 5, device=DEV)
for j, nt in enumerate([0.5, 0.6, 0.7, 0.8, 0.9]):
    nm, em = prune(nt)
    Ap = A * em; Ap[~nm] = 0; Ap[:, ~nm] = 0
    sweep.append([nt, int(nm[:NS].sum()), int(em.sum()), *scores(Ap)])
    level[nm[:NS] & (level == 5)] = j
    del Ap, em
    print(f"node threshold {nt}: {sweep[-1][1]:5,} features, {sweep[-1][2]:9,} edges; "
          f"replacement {sweep[-1][3]:.3f}, completeness {sweep[-1][4]:.3f}")
for (nt, nf, ne, r, c), lec in zip(sweep, [[0.5, 153, 9580, 0.5167, 0.8327], [0.6, 403, 39855, 0.5463, 0.8496], [0.7, 898, 158075, 0.5905, 0.8713], [0.8, 1848, 577275, 0.6437, 0.8943], [0.9, 3615, 1964282, 0.6849, 0.9115]]):
    check(f"features kept at node threshold {nt}", nf, lec[1], tol=5)

fig, ax = plt.subplots(figsize=(9, 5))
pos = torch.tensor([feats[i][1] for i in sel.tolist()]) + (torch.rand(NS) - 0.5) * 0.7
lay = torch.tensor([feats[i][0] for i in sel.tolist()]) + (torch.rand(NS) - 0.5) * 0.7
for j, (sz, sh) in enumerate([(14, INK), (9, INK), (6, INK), (4, INK2), (3, INK2), (1, "#C9BFAE")][::-1]):
    m = (level == 5 - j).cpu()
    ax.scatter(pos[m], lay[m], s=sz, color=sh, lw=0)
ax.set_xticks(range(T), [tok.decode(i) for i in ids[0]], rotation=40, fontsize=9)
ax.set_ylabel("layer"); ax.set_ylim(-1, 26)
ax.set_title(f"{NS:,} features; darker and larger = kept at a stricter pruning ({sweep[0][1]} survive 0.5)",
             loc="left", fontsize=10)
plt.show()

### How much of the graph is explained? (Plate XIII)

The **replacement score** asks how much of the words' influence on the answer reaches it through features rather than
through error nodes; the **completeness** score gives partial credit to nodes mostly explained by features.

In [ ]:
inf_ = influence(normed(A), w)
tot = inf_[:e1 + T].sum()
share = {"features": (inf_[:e0].sum() / tot).item(), "errors": (inf_[e0:e1].sum() / tot).item(),
         "embeddings": (inf_[e1:e1 + T].sum() / tot).item()}
err_austin = normed(A)[-NL, e0:e1].sum().item()
print(f"replacement {rep:.3f}: {1 - rep:.1%} of the words' influence on the answer runs through error nodes")
print("influence: " + ", ".join(f"{k} {v:.1%}" for k, v in share.items()))
print(f"error nodes supply {err_austin:.1%} of the Austin score's direct input")
check("replacement score", rep, 0.718, tol=0.003)
check("completeness score", comp, 0.9251, tol=0.003)
for k, v in share.items():
    check(f"influence share: {k}", v, {'features': 0.6384, 'errors': 0.102, 'embeddings': 0.2597}[k], tol=0.005)
check("error share of the Austin score's input", err_austin, 0.1654, tol=0.005)

The lecture reported the same: 28% of the words' influence runs through what the transcoders miss. The lecture's
cross-layer graph (not rebuilt here) scored 0.689, with 5.7 times fewer features.

---
## Part the Third: Dallas → Texas → Austin

### Naming the nodes (Plate XIV)

The lecture labelled each feature from its top contexts in 762,000 tokens of FineWeb-Edu (the *input side*; downloaded
here, not rerun) and from what it writes (the *output side*, which we compute).

In [ ]:
LAB = json.load(open(site("data/labels.json", "the lecture's feature labels")))["features"]
CARDS = {'15_4494': [' itſelf', ' Monfieur', ' Theſe', ' againſt'], '6_4662': [' city', ' CITY', ' City', 'city'], '4_7671': [' headquarters', ' Headquarters', 'Headquarters', ' HQ'], '3_13984': ['ⓧ', 'SizeMode', ' commerciales', '⎧'], '1_1000': [' stanovnika', ' européennes', ' εκεί', ' bulat'], '6_4012': [' statewide', 'MigrationBuilder', ' Statewide', ' estadual'], '0_13727': ['\u200d', ' year', 'bade', 'STIC'], '20_15589': [' TEXAS', ' Texas', 'Texas', ' AUSTIN'], '19_7477': ['GEBURTSDATUM', 'tvguidetime', ' <<<<<<<<<<<<<<', ' noqa'], '16_25': [' Texas', ' texas', ' Lubbock', ' Texan'], '4_13154': ['<bos>', ' Texas', 'Texas', 'TX'], '14_2268': [' Texas', ' Texan', 'Texas', ' Dallas'], '7_6861': ['ⓧ', 'PyExc', '".$_', 'bootstrapcdn'], '21_5943': [' Nairobi', ' Sydney', ' Dublin', ' London'], '17_7178': [' Capitol', ' Legislative', 'Capitol', ' Parliament'], '23_12237': ['ModelAdmin', ' Austin', "__':", "__':\r"]}
agree_cards = 0
for key, lec_words in CARDS.items():
    l, f = map(int, key.split("_"))
    ours_w = writes(l, f, 10)
    agree_cards += len(set(lec_words) & set(ours_w)) >= 3
    ex = LAB[key]["examples"][0] if key in LAB else None
    shown_ex = f"... {ex['left'][-30:]}[{ex['token']}]{ex['right']}" if ex else ""
    print(f"{key:9s} {LAB[key]['group'] if key in LAB else '':14s} writes {ours_w[:4]}  {shown_ex!r}")
check("cards whose top four written words we reproduce (at least 3 of 4)", agree_cards, len(CARDS))

Two of the six Texas features (19/7477 and 7/6861) fire on Texas but write unrelated tokens: a feature's two sides can
disagree.

### The two hops, and the shortcut (Plates XV–XVI)

Group the features into the lecture's **supernodes** (from the Neuronpedia annotation of this graph, which the lecture
checked against its own) and add up the edges between groups. The flows between features agree with the lecture's to
within a few percent. The error nodes' flows differ more (about 8%): an error node is whatever the transcoder misses, so
it also collects every rounding difference between the lecture's 16-bit run and this 32-bit one.

In [ ]:
SUPER = {"capital": [(15, 4, 4494), (6, 4, 4662), (4, 4, 7671), (3, 4, 13984), (1, 4, 1000)],
         "state": [(6, 7, 4012), (0, 7, 13727)], "Texas": TEXAS,
         "say a capital": [(21, 10, 5943), (17, 10, 7178), (7, 10, 691), (16, 10, 4298)],
         "say Austin": [(23, 10, 12237)], "place after a preposition": [(19, 10, 1445), (18, 10, 6101)],
         "Texas (at 'is')": [(20, 10, 15589)], "state government": [(18, 10, 8959)]}
grp = {k: [col[x] for x in v if x in col] for k, v in SUPER.items()}
check("annotated features in our graph", sum(len(v) for v in grp.values()), sum(len(v) for v in SUPER.values()))
grp["emb: Dallas"], grp["emb: capital"] = [e1 + 9], [e1 + 4]
grp["all errors"] = list(range(e0, e1))
grp["logit: Austin"] = [len(cols) - NL]
def flow(a, b):
    return A[grp[b]][:, grp[a]].sum().item()
LEC = {'emb: Dallas -> Texas': 242.375, 'emb: capital -> capital': 147.875, "Texas -> Texas (at 'is')": 42.2734, 'capital -> say a capital': 18.8509, 'emb: capital -> say a capital': 36.3125, 'say a capital -> say Austin': 13.8125, 'Texas -> say Austin': 10.6758, "Texas (at 'is') -> say Austin": 6.4062, 'emb: Dallas -> say Austin': 7.4062, 'all errors -> say Austin': 15.5072, 'all errors -> Texas': -52.5732}
for key, lec in LEC.items():
    a, b = key.split(" -> ")
    print(f"{key:40s} {flow(a, b):8.2f}   (lecture {lec:.2f})")
    # error nodes collect every rounding difference between the lecture's 16-bit run and this 32-bit one: 10% for them
    check(key, flow(a, b), lec, tol=max(1.5, abs(lec) * 0.1) if a == "all errors" else max(0.3, abs(lec) * 0.03))

In [ ]:
r_aus = A[len(cols) - NL]
parts = {k: r_aus[v].sum().item() for k, v in grp.items() if k != "logit: Austin"}
parts["sum_of_all_inputs"] = r_aus.sum().item()
for k, lec in {'all errors': 10.8816, 'Texas': 6.7285, "Texas (at 'is')": 5.125, 'emb: Dallas': 4.9375, 'say a capital': 3.1807, 'say Austin': 2.7188, 'sum_of_all_inputs': 33.8024}.items():
    print(f"into the Austin score, {k:22s} {parts[k]:7.2f}   (lecture {lec:.2f})")
    check(f"Austin score input: {k}", parts[k], lec, tol=0.35)
j = r_aus[:NS].argmin().item()
neg = feats[sel[j].item()]
print(f"strongest single push against Austin: {neg[0]}/{neg[2]} at word {neg[1]} ({r_aus[j]:.2f}), unlabelled")

# the two-hop picture, drawn from our numbers (the shortcut from the word "Dallas" runs along the left, dashed)
B = {"Dallas": (0.6, 0), "capital (word)": (3.4, 0), "Texas": (0.6, 1.6), "capital": (3.4, 1.6),
     "say a capital": (3.4, 3.2), "say Austin": (2.3, 4.4), "Austin": (0.9, 5.8)}
E2 = [("Dallas", "Texas", flow("emb: Dallas", "Texas")), ("capital (word)", "capital", flow("emb: capital", "capital")),
      ("capital", "say a capital", flow("capital", "say a capital")),
      ("Texas", "say Austin", flow("Texas", "say Austin") + flow("Texas (at 'is')", "say Austin")),
      ("say a capital", "say Austin", flow("say a capital", "say Austin")), ("say Austin", "Austin", parts["say Austin"]),
      ("Texas", "Austin", parts["Texas"] + parts["Texas (at 'is')"])]
fig, ax = plt.subplots(figsize=(7.5, 6.4))
def edge_label(x, y, v):
    ax.text(x, y, f"{v:.1f}", fontsize=8, color=INK2, ha="center", va="center", bbox=dict(fc=PAPER, ec="none", pad=0.6))
for a, b, v in E2:
    (x0, y0), (x1, y1) = B[a], B[b]
    ax.annotate("", (x1, y1), (x0, y0), arrowprops=dict(arrowstyle="-|>", color=INK, lw=0.6 + v / 25, shrinkA=15, shrinkB=15))
    n = math.hypot(x1 - x0, y1 - y0)
    edge_label(x0 + (x1 - x0) * 0.45 + 0.22 * (y1 - y0) / n, y0 + (y1 - y0) * 0.45 - 0.22 * (x1 - x0) / n, v)
for rail, b, v in ((-0.15, "say Austin", flow("emb: Dallas", "say Austin")), (-0.5, "Austin", parts["emb: Dallas"])):
    x1, y1 = B[b]
    ax.plot([0.1, rail, rail], [0, 0, y1], color=INK, lw=0.9, ls="--")
    ax.annotate("", (x1 - (0.52 if b == "say Austin" else 0.36), y1), (rail, y1),
                arrowprops=dict(arrowstyle="-|>", color=INK, lw=0.9, ls="--"))
    edge_label(rail, y1 * 0.55, v)
for k, (x, y) in B.items():
    ax.text(x, y, k, ha="center", va="center", fontsize=10, bbox=dict(boxstyle="round", fc=PAPER, ec=INK, lw=0.8))
ax.set_xlim(-0.8, 4.1); ax.set_ylim(-0.5, 6.3); ax.axis("off")
ax.set_title("Dallas → Texas → Austin, and the shortcut (dashed); summed edges", loc="left", fontsize=10)
plt.show()

Two hops: *Dallas* switches on Texas features, *capital* switches on capital features, and together they feed *say a
capital* and *say Austin* over the last word. And a shortcut: the word *Dallas* itself pushes toward *Austin* through
attention, without passing through Texas. Error nodes still supply about a third of the Austin score's direct input.

### Surgery: is the graph right? (Plate XVII)

A graph is a claim about the model. Test it on the real model: set a supernode's features to a multiple of their normal
activation (×0 turns them off, ×−2 pushes them the other way) and let everything downstream respond. As in the lecture
(circuit-tracer's default), attention patterns stay frozen at the prompt's own; the MLPs run for real, and each
intervened feature's change, times its decoder row, is added to its layer's MLP output.

In [ ]:
OTHER = {"Oakland": ("Fact: the capital of the state containing Oakland is",
                     {"California": [(22, 10, 4367), (21, 10, 2464), (6, 9, 13909), (8, 9, 14641), (14, 9, 12562)]}),
         "Shanghai": ("Fact: the capital of the country containing Shanghai is",
                      {"China": [(19, 9, 12274), (14, 9, 12274), (6, 9, 6811), (4, 9, 11570), (4, 9, 4257),
                                 (19, 10, 12274), (18, 10, 7639)]}),
         "Vancouver": ("Fact: the capital of the territory containing Vancouver is",
                       {"British Columbia": [(18, 10, 1025)], "Canada": [(14, 9, 5600), (6, 9, 11873), (4, 9, 8439)]})}
VEC = {}
def live(r, l, p, f):
    if (l, f) not in VEC:
        VEC[(l, f)] = feature(l, f)
    we, be, th, _ = VEC[(l, f)]
    x = r["u"][l][0, p] @ we + be
    return (x * (x > th)).item()

DEFAULT = {k: [(l, p, f, live(rec, l, p, f)) for l, p, f in v] for k, v in SUPER.items()}
for prompt, table in OTHER.values():
    r = {}
    with torch.no_grad():
        gemma(tok(prompt, return_tensors="pt").input_ids.to(DEV), rec=r)
    DEFAULT.update({k: [(l, p, f, live(r, l, p, f)) for l, p, f in v] for k, v in table.items()})

def intervene(ivs, freeze=True):
    """ivs: [(supernode, scale)]. Returns the top-5 next words and probabilities."""
    edits = {}
    for name, s in ivs:
        for l, p, f, a in DEFAULT[name]:
            edits.setdefault(l, []).append((p, f, s * a))
    def mlp(l, u, mo):
        if l not in edits:
            return mo
        mo = mo.clone()
        for p, f, val in edits[l]:
            we, be, th, wd = VEC[(l, f)]
            x = u[0, p] @ we + be
            mo[0, p] += (val - x * (x > th)) * wd          # set the feature to val: add the difference times its write
        return mo
    with torch.no_grad():
        pr = logits_of(gemma(ids, fz={"P": rec["P"]} if freeze else None, mlp=mlp))[0, -1].softmax(-1)
    v, i = pr.topk(5)
    return [(tok.decode(j), x) for j, x in zip(i.tolist(), v.tolist())], pr

LEC = {'say a capital×0': [[' Texas', 0.18358, 25.625], [' Austin', 0.09826, 25.0], [' the', 0.09826, 25.0]], 'say a capital×-2': [[' Texas', 0.4528, 27.0], [' the', 0.07868, 25.25], [' called', 0.04772, 24.75]], 'capital×0': [[' Austin', 0.26357, 25.75], [' Fort', 0.07552, 24.5], [' Texas', 0.07552, 24.5]], 'capital×-2': [[' Texas', 0.27968, 26.375], [' the', 0.0908, 25.25], [' called', 0.07071, 25.0]], 'Texas×0': [[' Austin', 0.18674, 25.375], [' the', 0.06063, 24.25], [' Oklahoma', 0.0535, 24.125]], 'Texas×-2': [[' the', 0.08163, 24.75], [' Albany', 0.07204, 24.625], [' Harrisburg', 0.06358, 24.5]], 'state×0': [[' Austin', 0.41124, 26.125], [' Texas', 0.05566, 24.125], [' the', 0.05566, 24.125]], 'state×-2': [[' Austin', 0.36917, 26.0], [' Texas', 0.07269, 24.375], [' the', 0.06415, 24.25]], 'say Austin×0': [[' Austin', 0.32132, 25.625], [' Texas', 0.08124, 24.25], [' the', 0.0717, 24.125]], 'say Austin×-2': [[' Texas', 0.17431, 24.5], [' Austin', 0.13576, 24.25], [' the', 0.0933, 23.875]], 'Texas×-2+California×2': [[' Sacramento', 0.35881, 26.25], [' the', 0.06235, 24.5], [' not', 0.04285, 24.125]], 'Texas×-2+California×2 (live attention)': [[' Sacramento', 0.26758, 26.125], [' the', 0.07666, 24.875], [' not', 0.03196, 24.0]], 'Texas×0+California×1': [[' Austin', 0.17707, 25.25], [' Sacramento', 0.1379, 25.0], [' not', 0.05749, 24.125]], 'Texas×-2+China×2': [[' Beijing', 0.20928, 25.625], [' not', 0.07699, 24.625], [' the', 0.06794, 24.5]], 'Texas×-2+British Columbia×2': [[' the', 0.07981, 24.75], [' Albany', 0.06216, 24.5], [' not', 0.06216, 24.5]], 'Texas×-2+British Columbia×6': [[' not', 0.10344, 24.75], [' the', 0.08056, 24.5], [' Augusta', 0.07109, 24.375]], 'Texas×-2+Canada×2+British Columbia×2': [[' Ottawa', 0.11083, 25.125], [' the', 0.07617, 24.75], [' Toronto', 0.06722, 24.625]]}
def label(ivs, freeze=True):
    return "+".join(f"{n}×{s:g}" for n, s in ivs) + ("" if freeze else " (live attention)")
singles = [[(n, s)] for n in ("say a capital", "capital", "Texas", "state", "say Austin") for s in (0.0, -2.0)]
fig, axes = plt.subplots(2, 5, figsize=(13, 4.6), sharex=True)
for ax, ivs in zip(axes.T.flatten(), singles):
    top5, _ = intervene(ivs)
    lec = LEC[label(ivs)]
    ax.barh(range(5)[::-1], [p for _, p in top5], color=INK, height=0.6)
    ax.set_yticks(range(5)[::-1], [w for w, _ in top5], fontsize=8)
    ax.set_title(minus(label(ivs)), fontsize=9, loc="left")
    check(minus(f"{label(ivs)}: top word"), top5[0][0], lec[0][0])
    check(minus(f"{label(ivs)}: its probability"), top5[0][1], lec[0][1], tol=0.03)
plt.tight_layout(); plt.show()

Turn *say a capital* down and Gemma says the state instead; turn Texas the other way and it names other states'
capitals; *state* barely matters. One surprise the lecture flags: Texas at ×0 is not enough. Austin stays on top (at
18%), because other routes (the shortcut, the error nodes) still carry it; it takes negative steering.

### The swap (Plate XVIII)

Texas down, California up (California's features taken from the Oakland prompt, at their activations there). The
lecture's two dials, as a grid of 49 settings.

In [ ]:
TS_, CS_ = [1.0, 0.5, 0.0, -0.5, -1.0, -1.5, -2.0], [0.0, 0.5, 1.0, 1.5, 2.0, 3.0, 4.0]
sac, aus = tok.encode(" Sacramento", add_special_tokens=False)[0], tok.encode(" Austin", add_special_tokens=False)[0]
GRID = {(g[0], g[1]): g for g in [[1.0, 0.0, ' Austin', 0.005309, 0.421769], [1.0, 0.5, ' Austin', 0.012046, 0.398903], [1.0, 1.0, ' Austin', 0.02698, 0.372454], [1.0, 1.5, ' Austin', 0.056974, 0.327866], [1.0, 2.0, ' Austin', 0.075696, 0.299385], [1.0, 3.0, ' Austin', 0.156403, 0.200825], [1.0, 4.0, ' Sacramento', 0.216483, 0.10226], [0.5, 0.0, ' Austin', 0.014434, 0.328523], [0.5, 0.5, ' Austin', 0.032836, 0.311542], [0.5, 1.0, ' Austin', 0.072525, 0.286842], [0.5, 1.5, ' Austin', 0.125303, 0.234097], [0.5, 2.0, ' Austin', 0.182691, 0.207016], [0.5, 3.0, ' Sacramento', 0.264304, 0.110178], [0.5, 4.0, ' Sacramento', 0.307309, 0.04159], [0.0, 0.0, ' Austin', 0.036025, 0.182949], [0.0, 0.5, ' Austin', 0.082033, 0.173664], [0.0, 1.0, ' Austin', 0.137899, 0.177066], [0.0, 1.5, ' Sacramento', 0.19905, 0.136805], [0.0, 2.0, ' Sacramento', 0.292358, 0.083762], [0.0, 3.0, ' Sacramento', 0.394374, 0.028568], [0.0, 4.0, ' Sacramento', 0.439672, 0.008053], [-0.5, 0.0, ' the', 0.050995, 0.021258], [-0.5, 0.5, ' Sacramento', 0.141463, 0.024583], [-0.5, 1.0, ' Sacramento', 0.208732, 0.022], [-0.5, 1.5, ' Sacramento', 0.330917, 0.016475], [-0.5, 2.0, ' Sacramento', 0.417785, 0.008671], [-0.5, 3.0, ' Sacramento', 0.493101, 0.002932], [-0.5, 4.0, ' Sacramento', 0.484819, 0.000936], [-1.0, 0.0, ' Harrisburg', 0.041164, 0.000587], [-1.0, 0.5, ' Sacramento', 0.151214, 0.000793], [-1.0, 1.0, ' Sacramento', 0.223091, 0.000805], [-1.0, 1.5, ' Sacramento', 0.374979, 0.000724], [-1.0, 2.0, ' Sacramento', 0.460136, 0.000475], [-1.0, 3.0, ' Sacramento', 0.521817, 0.000198], [-1.0, 4.0, ' Sacramento', 0.525075, 7.3e-05], [-1.5, 0.0, ' Harrisburg', 0.03311, 4.4e-05], [-1.5, 0.5, ' Sacramento', 0.120876, 5.9e-05], [-1.5, 1.0, ' Sacramento', 0.206738, 6.1e-05], [-1.5, 1.5, ' Sacramento', 0.316774, 7.3e-05], [-1.5, 2.0, ' Sacramento', 0.422556, 5.9e-05], [-1.5, 3.0, ' Sacramento', 0.50599, 2.3e-05], [-1.5, 4.0, ' Sacramento', 0.547879, 8e-06], [-2.0, 0.0, ' the', 0.029553, 8e-06], [-2.0, 0.5, ' Sacramento', 0.085116, 9e-06], [-2.0, 1.0, ' Sacramento', 0.18814, 1.2e-05], [-2.0, 1.5, ' Sacramento', 0.26935, 1e-05], [-2.0, 2.0, ' Sacramento', 0.358806, 1.1e-05], [-2.0, 3.0, ' Sacramento', 0.4637, 5e-06], [-2.0, 4.0, ' Sacramento', 0.541234, 2e-06]]}
psac, top1, same_top = np.zeros((7, 7)), {}, 0
for i, ts in enumerate(TS_):
    for j, cs in enumerate(CS_):
        top5, pr = intervene([("Texas", ts), ("California", cs)])
        psac[i, j] = pr[sac].item()
        top1[(ts, cs)] = top5[0][0]
        same_top += top5[0][0] == GRID[(ts, cs)][2]
fig, ax = plt.subplots(figsize=(6.5, 4.6))
from matplotlib.colors import LinearSegmentedColormap
ax.imshow(psac, cmap=LinearSegmentedColormap.from_list("ink", [PAPER, INK]), vmin=0, vmax=0.6)
for i, ts in enumerate(TS_):
    for j, cs in enumerate(CS_):
        ax.text(j, i, top1[(ts, cs)].strip()[:6], ha="center", va="center", fontsize=7,
                color=PAPER if psac[i, j] > 0.3 else INK)
ax.set_xticks(range(7), [f"×{c:g}" for c in CS_]); ax.set_yticks(range(7), [minus(f"×{t:g}") for t in TS_])
ax.set_xlabel("California"); ax.set_ylabel("Texas")
ax.set_title("top word; shading = P(Sacramento)", loc="left", fontsize=10)
plt.show()
swap_top5, _ = intervene([("Texas", -2.0), ("California", 2.0)])
print("Texas ×−2, California ×2:", [(w_, round(p_, 3)) for w_, p_ in swap_top5])
check("grid cells whose top word matches the lecture's", same_top, 49, tol=2)
check("P(Sacramento) at Texas ×−2, California ×2", psac[6, 4], GRID[(-2.0, 2.0)][3], tol=0.02)
check("the highest P(Sacramento) in the grid", psac.max(), max(g[3] for g in GRID.values()), tol=0.03)

### Where the swap fails (Plate XX)

The same surgery with China (from *the country containing Shanghai*) works. British Columbia, the swap Anthropic
reported in Claude 3.5 Haiku, does not work in Gemma, at ×2 or ×6; add Canada's features and Gemma answers with a
country's capital.

In [ ]:
LEC = {'say a capital×0': [[' Texas', 0.18358, 25.625], [' Austin', 0.09826, 25.0], [' the', 0.09826, 25.0]], 'say a capital×-2': [[' Texas', 0.4528, 27.0], [' the', 0.07868, 25.25], [' called', 0.04772, 24.75]], 'capital×0': [[' Austin', 0.26357, 25.75], [' Fort', 0.07552, 24.5], [' Texas', 0.07552, 24.5]], 'capital×-2': [[' Texas', 0.27968, 26.375], [' the', 0.0908, 25.25], [' called', 0.07071, 25.0]], 'Texas×0': [[' Austin', 0.18674, 25.375], [' the', 0.06063, 24.25], [' Oklahoma', 0.0535, 24.125]], 'Texas×-2': [[' the', 0.08163, 24.75], [' Albany', 0.07204, 24.625], [' Harrisburg', 0.06358, 24.5]], 'state×0': [[' Austin', 0.41124, 26.125], [' Texas', 0.05566, 24.125], [' the', 0.05566, 24.125]], 'state×-2': [[' Austin', 0.36917, 26.0], [' Texas', 0.07269, 24.375], [' the', 0.06415, 24.25]], 'say Austin×0': [[' Austin', 0.32132, 25.625], [' Texas', 0.08124, 24.25], [' the', 0.0717, 24.125]], 'say Austin×-2': [[' Texas', 0.17431, 24.5], [' Austin', 0.13576, 24.25], [' the', 0.0933, 23.875]], 'Texas×-2+California×2': [[' Sacramento', 0.35881, 26.25], [' the', 0.06235, 24.5], [' not', 0.04285, 24.125]], 'Texas×-2+California×2 (live attention)': [[' Sacramento', 0.26758, 26.125], [' the', 0.07666, 24.875], [' not', 0.03196, 24.0]], 'Texas×0+California×1': [[' Austin', 0.17707, 25.25], [' Sacramento', 0.1379, 25.0], [' not', 0.05749, 24.125]], 'Texas×-2+China×2': [[' Beijing', 0.20928, 25.625], [' not', 0.07699, 24.625], [' the', 0.06794, 24.5]], 'Texas×-2+British Columbia×2': [[' the', 0.07981, 24.75], [' Albany', 0.06216, 24.5], [' not', 0.06216, 24.5]], 'Texas×-2+British Columbia×6': [[' not', 0.10344, 24.75], [' the', 0.08056, 24.5], [' Augusta', 0.07109, 24.375]], 'Texas×-2+Canada×2+British Columbia×2': [[' Ottawa', 0.11083, 25.125], [' the', 0.07617, 24.75], [' Toronto', 0.06722, 24.625]]}
for ivs in ([("Texas", -2.0), ("China", 2.0)], [("Texas", -2.0), ("British Columbia", 2.0)],
            [("Texas", -2.0), ("British Columbia", 6.0)], [("Texas", -2.0), ("Canada", 2.0), ("British Columbia", 2.0)]):
    top5, _ = intervene(ivs)
    print(f"{minus(label(ivs)):45s}", [(w_, round(p_, 3)) for w_, p_ in top5[:3]])
    check(minus(f"{label(ivs)}: top word"), top5[0][0], LEC[label(ivs)][0][0])
    check(minus(f"{label(ivs)}: its probability"), top5[0][1], LEC[label(ivs)][0][1], tol=0.03)

### Does it hold beyond one prompt? (Plate XXI)

The Texas features' summed activation (each feature's largest over the prompt) on all 35 two-hop prompts, and on six
rewordings of the Dallas question.

In [ ]:
TXG = TEXAS + [(20, 10, 15589)]                                   # the lecture's sum counts 20/15589 twice
def texas_group(prompt):
    r = {}
    with torch.no_grad():
        i = tok(prompt, return_tensors="pt").input_ids.to(DEV)
        lg_ = logits_of(gemma(i, rec=r))[0, -1].softmax(-1)
    n = i.shape[1]
    g = 0.0
    for l, _, f in TXG:
        we, be, th, _ = VEC[(l, f)]
        x = r["u"][l][0, 1:n] @ we + be
        g += (x * (x > th)).max().item()
    return g, lg_
CG = {'Dallas': 235.688, 'Houston': 189.375, 'San Antonio': 231.0, 'El Paso': 175.125, 'Oakland': 1.781, 'Los Angeles': 1.781, 'San Francisco': 1.781, 'San Diego': 1.781, 'Miami': 1.781, 'Chicago': 1.781, 'Seattle': 1.781, 'Buffalo': 1.781, 'Detroit': 1.781, 'Philadelphia': 1.781, 'Pittsburgh': 1.781, 'Las Vegas': 1.781, 'Portland': 1.781, 'Milwaukee': 1.781, 'New Orleans': 1.781, 'Kansas City': 1.781, 'Cleveland': 1.781, 'Savannah': 1.781, 'Baltimore': 1.781, 'Louisville': 1.781, 'Memphis': 1.781, 'Tucson': 1.781, 'Birmingham': 1.781, 'Burlington': 1.781, 'Charleston': 1.781, 'Anchorage': 1.781, 'Omaha': 1.781, 'Wilmington': 1.781, 'Newark': 1.781, 'Minneapolis': 1.781, 'Spokane': 1.781}
vals = {c: texas_group(f"Fact: the capital of the state containing {c} is")[0] for c, _, _ in CITIES}
fig, ax = plt.subplots(figsize=(11, 3))
ax.bar(range(35), [vals[c] for c, _, _ in CITIES], color=[INK if s == "Texas" else INK2 for _, s, _ in CITIES])
ax.set_xticks(range(35), [c for c, _, _ in CITIES], rotation=70, fontsize=8)
ax.set_ylabel("Texas features, summed"); plt.show()
for c in ("Dallas", "San Antonio", "Houston", "El Paso", "Chicago"):
    check(f"Texas group on {c}", vals[c], CG[c], tol=max(1.5, CG[c] * 0.01))
for prompt, lec_w, lec_p in [['Fact: the capital of the state containing Dallas is', ' Austin', 0.41321], ['The capital of the state where Dallas is located is', ' Austin', 0.32016], ['Dallas is a city in a state. The capital of that state is', ' Austin', 0.49528], ['Q: What is the capital of the state that Dallas is in?\nA:', ' Austin', 0.77008], ['Fact: the state capital of the state containing Dallas is', ' Austin', 0.49071], ['Fact: Dallas is in a state whose capital is', ' Austin', 0.33804]]:
    g, pr = texas_group(prompt)
    print(f"{prompt!r:62}  Austin {pr[aus]:.1%}  (lecture {lec_p:.1%}), Texas group {g:.0f}")
    check("the top word", tok.decode(pr.argmax()), lec_w)

The Texas features are about **Texas**, not Dallas: the four Texas cities light them up (175–236), and every other city
leaves them at the same small value. Every rewording still answers Austin.

### What the graph cannot tell us (Plate XXII)

- **Errors.** 28% of the words' influence on the answer goes through what the transcoders miss, and error nodes supply a
  third of the Austin score's direct input.
- **The copy on its own.** With no error nodes, per-layer transcoders agree with Gemma's top word 34.5% of the time
  (Gemma's cross-layer transcoder, in the lecture: 51.1%).
- **Frozen attention.** The graph holds attention fixed. Let it move during the swap and Sacramento drops:

In [ ]:
froz, _ = intervene([("Texas", -2.0), ("California", 2.0)])
liv, _ = intervene([("Texas", -2.0), ("California", 2.0)], freeze=False)
print(f"P(Sacramento): attention frozen {froz[0][1]:.1%}, attention live {liv[0][1]:.1%} (top word {liv[0][0]!r})")
check("swap with live attention: P(Sacramento)", liv[0][1], 0.26758, tol=0.015)
print(f"\nthe notebook so far: {(time.time() - T0) / 60:.1f} minutes")

- **One prompt.** A graph explains one prompt; Anthropic report that they could get satisfying graphs for about a quarter
  of the prompts they tried (their number).

---
## Your turn

1. **Thirteen layers.** Swap only the first 13 MLPs on the 64 test documents. How often does the copy agree with Gemma?
   *Hint: `compare(h_real, gemma(test, mlp=swap(set(range(13)))), test)`.*
2. **The capital hop.** What is the summed direct effect of the *capital* supernode on *say a capital*?
   *Hint: `flow(a, b)`.*
3. **A gentler push.** Texas at ×−0.5, nothing else. What is P(Austin)?
   *Hint: `intervene(...)` returns the full probabilities too; `aus` is Austin's token id.*
4. **Against Austin.** Which single node pushes hardest *against* the Austin score, among all its direct inputs
   (features, error nodes and words)? *Hint: `r_aus.argmin()`, and the column layout `e0`, `e1`.*
5. **How big is the pruned graph?** At node threshold 0.8, how many edges survive?
   *Hint: `sweep`.*

In [ ]:
# Your turn 1

In [ ]:
# Your turn 2

In [ ]:
# Your turn 3

In [ ]:
# Your turn 4

In [ ]:
# Your turn 5

---
## Solutions

In [ ]:
# 1. Thirteen layers
with torch.no_grad():
    r13, _ = compare(h_real, gemma(test, mlp=swap(set(range(13)))), test)
print(f"first 13 MLPs swapped: {r13['agree']:.1%} agreement")
check("top-1 agreement, first 13 MLPs swapped", r13["agree"], 0.7183, tol=0.006)

In [ ]:
# 2. The capital hop
print(f"capital -> say a capital: {flow('capital', 'say a capital'):.2f} (plus {flow('emb: capital', 'say a capital'):.2f} "
      "straight from the word 'capital')")
check("capital -> say a capital", flow("capital", "say a capital"), 18.8509, tol=0.6)

In [ ]:
# 3. A gentler push
top5, pr = intervene([("Texas", -0.5), ("California", 0.0)])
print(f"Texas ×−0.5: P(Austin) {pr[aus]:.1%}; top word {top5[0][0]!r}")
check("P(Austin) at Texas ×−0.5", pr[aus].item(), 0.021258, tol=0.01)

In [ ]:
# 4. Against Austin
j = r_aus.argmin().item()
who_ = f"feature {feats[sel[j].item()][0]}/{feats[sel[j].item()][2]} at word {feats[sel[j].item()][1]}" if j < e0 else (
       f"error node, layer {(j - e0) // T}, word {(j - e0) % T}" if j < e1 else f"the word {tok.decode(ids[0, j - e1])!r}")
print(f"the strongest push against Austin: {who_}, {r_aus[j]:.2f}")
check("it is 24/6044 at 'is'", (j < e0) and feats[sel[j].item()][:3] == (24, 10, 6044), True)
check("its push", r_aus[j].item(), -4.3438, tol=0.3)

In [ ]:
# 5. How big is the pruned graph?
print(f"node threshold 0.8: {sweep[3][1]:,} features and {sweep[3][2]:,} edges, of {(A != 0).sum().item():,} edges unpruned")
check("edges kept at 0.8", sweep[3][2], 577275, tol=6000)
print(f"\nthe whole notebook: {(time.time() - T0) / 60:.1f} minutes")